In [14]:
from typing import List, Any
import os
import weaviate
import json
import pandas as pd
from langchain_weaviate import WeaviateVectorStore
from sentence_transformers import SentenceTransformer
from IPython.display import display, Markdown
from dotenv import load_dotenv


In [2]:
load_dotenv('.env.example/.env')

# weaviate Keys
WEAVIATE_URL = os.environ["WEAVIATE_URL"]
WEAVIATE_API_KEY = os.environ["WEAVIATE_API_KEY"]

In [3]:
weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=WEAVIATE_API_KEY,
)

In [4]:
class SentenceTransformersEmbeddings:
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # returns a list of embeddings for documents
        return self.model.encode(texts).tolist()

    def embed_query(self, text: str) -> List[float]:
        # returns a single embedding for a query
        return self.model.encode([text])[0].tolist()

In [5]:
embedding_model = SentenceTransformersEmbeddings('sentence-transformers/all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 576.41it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
import pandas as pd

In [7]:
laws = pd.read_csv('dataset/act_raw_text_with_4meta.csv')

In [17]:
laws.columns


Index(['Unnamed: 0', 'CELEX', 'Status', 'Act_type', 'Treaty', 'act_raw_text'], dtype='str')

In [8]:
def get__embeddings(enhanced_query):
    query_embedding = embedding_model.embed_query(enhanced_query)

    return query_embedding

In [9]:
query_embedding = get__embeddings("driving without license penalty")

In [10]:
weaviate_client.is_live()

True

In [28]:
eu = weaviate_client.collections.use("Euro_Laws")
response = eu.query.near_vector(
    near_vector= query_embedding, 
    limit=5
)

for obj in response.objects:
    print(json.dumps(obj.properties, indent=10))  # Inspect the results

KeyboardInterrupt: 

In [11]:
vectorstore = WeaviateVectorStore(
    client = weaviate_client,
    index_name = "Euro_Laws",
    text_key="text",
    embedding = embedding_model
)

In [26]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("driving without license penalty")
docs

ValueError: Error during query: Query call with protocol GRPC search failed with message Deadline Exceeded.

```
'celex': , 'status': 

'act_type' , 'treaty':

```

In [15]:
def get_celex_ids(docs):
    celex_ids = []
    for i in docs:
        celex_ids.append(i.metadata['celex'])

    celex_ids = list(set(celex_ids))

    return celex_ids

get_celex_ids(docs)  

['32015L0413', '32006L0126', '32009R1072', '31980L1263']

In [22]:
laws[laws['CELEX'] == '32015L0413']['Status'].iloc[0]

'In Force'

In [18]:
def search_docs(query):
    celex_ids = []
    full_doc_info = """ """

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)

    for doc in docs:
        celex_ids.append(doc.metadata['celex'])

    celex_ids = list(set(celex_ids))    

    for i , celex_id in enumerate(celex_ids):

        full_doc_info += f"""

        doc {i} :

        'celex': {laws[laws['CELEX'] == celex_id]['CELEX'].iloc[0]}
        'status': {laws[laws['CELEX'] == celex_id]['Status'].iloc[0]}
        'act_type': {laws[laws['CELEX'] == celex_id]['Act_type'].iloc[0]}
        'treaty': {laws[laws['CELEX'] == celex_id]['Treaty'].iloc[0]}

        full_doc :

        {laws[laws['CELEX'] == celex_id]['act_raw_text'].iloc[0]}
{"=="*15} "END OF DOC" {"=="*15}
        """

    return full_doc_info    



In [20]:
display(Markdown(search_docs("driving without license penalty"))) 

 

        doc 0 :

        'celex': 31980L1263
        'status': Not in Force
        'act_type': Directive
        'treaty': TEEC

        full_doc :

        Avis juridique important|31980L1263First Council Directive 80/1263/EEC of 4 December 1980 on the introduction of a Community driving licence Official Journal L 375 , 31/12/1980 P. 0001 - 0015 Finnish special edition: Chapter 7 Volume 2 P. 0171 Greek special edition: Chapter 13 Volume 10 P. 0089 Swedish special edition: Chapter 7 Volume 2 P. 0171 Spanish special edition: Chapter 07 Volume 2 P. 0259 Portuguese special edition Chapter 07 Volume 2 P. 0259 FIRST COUNCIL DIRECTIVE of 4 December 1980 on the introduction of a Community driving licence (80/1263/EEC) THE COUNCIL OF THE EUROPEAN COMMUNITIES, Having regard to the Treaty establishing the European Economic Community, and in particular Article 75 (1) (c) thereof, Having regard to the proposal from the Commission, Having regard to the opinion of the European Parliament (1), Having regard to the opinion of the Economic and Social Committee (2), Whereas, for the purposes of the common transport policy, as a contribution to improving road traffic safety, and to assist the movement of persons settling in a Member State other than that in which they have passed a driving test, or moving within the Community, it is desirable that a Community driving licence be introduced; Whereas the introduction of a Community driving licence presupposes the harmonization of existing national driving test arrangements, which can only be achieved gradually ; whereas the first stage of this harmonization could culminate in the establishment of a Community model national licence and the mutual recognition by Member States of national driving licences and the exchange of licences by holders transferring their place of residence or place of employment from one Member State to another; Whereas the Community model national licence should be based on that defined by the Final Act of the Convention on Road Traffic drawn up in Vienna in November 1968 by the United Nations Road Traffic Conference; Whereas the mutual recognition of driving licences issued by the different Member States and the exchange of a licence by a holder moving from one Community country to reside or work in another will only be possible further to an initial harmonization of the regulations governing the issue and validity of licences; Whereas, without prejudice to the final provisions to be adopted by the Council on vehicle categories, it is necessary to establish common standards in respect of the validity of the licence for driving the different categories of vehicles, so that the Community model licence can be issued throughout the Community under comparable conditions; Whereas, however, at this initial harmonization stage and pending the introduction of the final system, Member States should be allowed to lay down the conditions with regard to age and the period of validity of licences arid also, under certain specific conditions, to derogate from the categories, speeds and conditions of validity laid down by this (1)OJ No C 238, 11.10.1976, p. 43. (2)OJ No C 197, 23.8.1976, p. 32. Directive ; and, where appropriate, to check the additional conditions laid down for the exchange of driving licences of certain categories of vehicles; Whereas it is desirable that the standards for testing drivers and issuing licences should be further harmonized as soon as possible, HAS ADOPTED THIS DIRECTIVE: Article 1 The Member States shall introduce a national driving licence based on the Community model provided for in Article 2. A Community model driving licence shall, subject to Article 8, entitle the holder to drive, both on national and international journeys, vehicles of the categories for which it has been granted. Community model driving licences shall be issued by the Member States in accordance with this Directive. Article 2 The driving licence provided for in Article 1 shall conform to the model in Annex I. The oval on page 1 of the model shall contain the distinguishing sign of the State issuing the licence. After consulting the Commission, Member States may adapt the model in the Annex in any way necessary to enable them to: - process the driving licence by computer, - enter in the licence any categories of vehicle which, pursuant to Article 9, differ from those provided for in Article 3. Member States shall take all necessary steps to avoid any risk of forgery of driving licences. Article 3 1. Without prejudice to the final provisions to be adopted by the Council concerning vehicle categories, the driving licence provided for in Article I shall authorize the driving on public roads of vehicles in the following categories: category A : motorcycles with or without side-car; category B : motor vehicles, other than those in category A, with a permissible maximum weight not exceeding 3 500 kg and not more than eight seats in addition to the driver's seat; category C : motor vehicles used for the carriage of goods and whose permissible maximum weight exceeds 3 500 kg; category D : motor vehicles used for the carriage of passengers, with more than eight seats in addition to the driver's seat: category E : combinations of vehicles of which the tractor vehicle is in a category or categories for which the driver is licensed (B and/or C and/or D), but which are not themselves in that category or categories. 2. For the purposes of paragraph 1: (a) a trailer with a permissible maximum weight not exceeding 750 kg may be coupled to a motor vehicle in category B above ; a trailer with a permissible maximum weight exceeding 750 kg may likewise be coupled to such vehicle, provided that the following two conditions are fulfilled: - the permissible maximum weight of the trailer does not exceed the unladen weight of the motor vehicle, and - the total permissible maximum weight of the combination of vehicles does not exceed 3 500 kg; (b) a motor vehicle in category C or D may be coupled to a trailer the permissible maximum weight of which does not exceed 750 kg. 3. For the purposes of this Article: - "motorcycle" means any two or three-wheeled vehicle with a maximum design speed exceeding 50 kph (33 mph) or, if it is powered by a heat propulsion engine, with a cylinder capacity exceeding 50 cc. In addition, in the case of a three-wheeled vehicle, the unladen weight shall not exceed 400 kg; - "power-driven vehicle" means any self-propelled vehicle running on a road, other than a railborne vehicle; - "motor vehicle" means any power-driven vehicle, other than a motorcycle, which is normally used for carrying persons or goods by road or for drawing, on the road, vehicles used for the carriage of persons or goods. This term shall include trolley buses, i.e. vehicles connected to an electric conductor and not rail borne. It shall not include agricultural or forestry tractors; - "agricultural or forestry tractor" means any power-driven vehicle running on wheels or tracks, having at least two axles, the principal function of which lies in its tractive power, which is specially designed to pull, push, carry or operate certain tools, machines or trailers used in connection with agricultural or forestry operations, and the use of which for carrying persons or goods by road or for drawing, on the road, vehicles used for the carriage or persons or goods is only a secondary function. Article 4 1. The validity of the driving licence provided for in Article 1 shall be determined as follows: (a) licences granted for categories C and D shall also be valid for the driving of vehicles in category B; (b) licences granted for category E shall, without prejudice to the provisions of (c), be valid for the driving of combinations of vehicles; (c) licences for category E shall be granted only to drivers already entitled to drive vehicles in category B, C or D. 2. Licences issued to disabled drivers shall specifically mention the conditions under which such drivers are entitled to drive. Article 5 1. Without prejudice to Article 5 of Council Regulation (EEC) No 543/69 of 25 March 1969 on the harmonization of certain social legislation relating to road transport (1), Member States shall fix the minimum age at which driving licences may be issued. 2. Member States may refuse to recognize the validity on their territory of driving licences issued to drivers under the age of 18 years. Article 6 1. A driving licence shall, moreover, be issued only to those applicants: (a) who have passed a practical and theoretical test and who meet medical standards, the minimum requirements of which may not be substantially less stringent than those set out in Annexes II and III; (b) who have their normal residence in the territory of the Member State issuing the licence, if the legislation of the Member State concerned so requires. 2. Member States may apply to the issue of driving licences the provisions of their national legislation relating thereto which are concerned with conditions other than those referred to in paragraph 1. Article 7 Without prejudice to the provisions which may be adopted by the Council in this regard, each Member State shall retain the right to fix, on the basis of national criteria, the period of validity of the driving licences (Community model) which it issues or exchanges pursuant to Article 8. Article 8 1. The Member States shall provide that, if the holder of a valid national driving licence or valid Community model licence issued by a Member State takes up normal residence in another Member State his licence shall remain valid there for up to a maximum of a year following the taking up of residence At the request of the holder within that period, and against surrender of his licence, the State in which he has taken up normal residence shall issue him with a driving licence (Community model) for the corresponding category or categories without subjecting him to the conditions laid down in Article 6. However, that Member State may refuse to exchange the licence if its national regulations, including medical standards, preclude the issue of the licence. The exchange must be preceded by the submission of a statement by the applicant to the effect that his (1)OJ No L 77, 29.3.1969, p. 49. driving licence is currently valid. It shall be for the Member State effecting the exchange to check the veracity of his statement if necessary. The Member State effecting the exchange shall return the old licence to the authorities of the Member State which issued it. 2. Member States which, pursuant to Article 9, do not apply categories C, D and E as defined in Article 3 (1) may: - exchange category C, D and E driving licences in accordance with paragraph 1 of this Article or, - require the applicant to furnish proof of driving experience and in this case issue a licence entitling him to drive vehicles in the national category in respect of which he furnished proof of adequate experience, or vehicles in a lower category. In any event, such States shall issue to the applicant at least a licence to drive vehicles in the lowest of the national categories corresponding to categories C, D and E as defined in Article 3 (1). During the year following the taking up of residence by drivers who have not applied for a licence exchange, such States shall recognize such drivers' licences as being equivalent at least to licences for the lowest relevant national category. 3. Where a Member State exchanges a licence, issued by a third country, for a Community model driving licence, such exchange shall be recorded in the licence, as shall any subsequent renewal or replacement of that licence. In the event of subsequent exchange of the said licence, Member States shall not be obliged to apply paragraph 1. A Community model driving licence may in any event be issued only if the licence issued by the third country has been surrendered to the competent authorities of the Member State issuing the Community licence. Article 9 After consulting the Commission, Member States may, pending introduction of the final system and provided that the fact is recorded on the licence, derogate from: - the categories defined in Article 3 (1); - the speeds indicated in the first indent of Article 3 (3), provided that the speeds which they prescribe are lower; - the conditions of validity provided for in Article 4. Furthermore, Member States shall, pursuant to the procedure laid down in Article 12, establish equivalent definitions in so far as their national categories differ. Article 10 The Council, acting on a proposal from the Commission, shall carry out as soon as possible a more detailed harmonization of the standards for driving tests and licensing with a view to inter alia subsequent improvements in road safety throughout the Community. Article 11 The Member States shall determine the arrangements for replacing currently valid national driving licences issued by them with Community model driving licences for the corresponding category or categories. This operation shall take place without the need for the tests provided for under Article 6, on submission of and in exchange for the old licences. Article 12 1. After consulting the Commission, Member States shall, in good time and at the latest by 30 June 1982, adopt the laws, regulations or administrative provisions necessary for the implementation of this Directive from 1 January 1983. 2. However, a Member State may, without prejudice to the application of the other provisions in this Directive, decide not to issue Community model driving licences until a later date, which may not be later than 1 January 1986. 3. Member States shall assist one another in the implementation of this Directive. Article 13 This Directive is addressed to the Member States. Done at Brussels, 4 December 1980. For the Council The President J. BARTHEL ANNEX I >PIC FILE= "T0014238">Comments on the model driving licence shown on page 1 1. The colour of the Community driving licence shall be pink. 2. On the cover page: - mention of the name of the Member State issuing the licence shall be optional, - the distinguishing sign of the Member State issuing the licence shall be entered in the oval, - the words "driving licence" shall be printed in large type in the language or languages of the Member State issuing the licence. They shall appear, alter a suitable space, in small type in the other languages of the European Communities, - the words "European Communities model" shall be printed in the language or languages of the Member State issuing the licence. 3. The printed entries on the other pages shall be in the language or languages of the Member State issuing the licence. 4. The page entitled "Additional information" is designed for details of any restriction or extension of the conditions governing the validity of the licence. This page may also be used for showing the period of validity of the licence where this varies. >PIC FILE= "T0014239"> 5. Other comments may be entered on the remaining blank pages. Where appropriate, Member States may enter on them categories of vehicles not covered by this Directive or may subdivide categories A, B, C, D and E in the corresponding page. 6. Member States shall have the right to: - dispense with the photograph requirement; - replace the permanent place of residence by the postal address; - delete the date of issue and indicate the date of commencement of validity of the licence. SPECIMEN COMMUNITY MODEL LICENCE : BELGIAN LICENCE (FOR INFORMATION) >PIC FILE= "T0014240"> ANNEX II MINIMUM REQUIREMENTS FOR DRIVING TESTS THEORETICAL TEST Form 1. The form chosen shall be such as to establish whether the candidate has the required knowledge and understanding of the subjects listed in paragraphs 2 and 3 of this Annex. Content 2. Knowledge and understanding of the regulations, and more especially of the rules applicable to the use of vehicles of the category corresponding to the type of licence applied for: 2.1. Knowledge and understanding of traffic rules and regulations, signs, signals and road markings and of their meaning; 2.2. Basic knowledge and understanding of the technical regulations relating to vehicle safety in traffic; 2.3. Knowledge and understanding of rules relating to the driver, in so far as they concern road safety, including, for drivers of category C and D vehicles only, rules relating to hours of work and rest periods; 2.4. Knowledge and understanding of the rules on what a driver should do in the event of an accident. 3. Knowledge and understanding of other subjects: 3.1. Adequate knowledge and understanding of the importance of road safety matters, and especially of the following accident factors: 3.1.1. Driving hazards, such as the danger of overtaking, misjudgement of speed (effects on braking and safety distances), influence of the weather (snow, rain, fog, side-winds, aquaplaning), behaviour of other road users, and in particular of elderly people and children; 3.1.2. Factors likely to reduce the driver's vigilance and his physical and mental fitness to drive, such as fatigue, illness, alcohol and other drugs, etc.; 3.1.3. Safety factors relating to vehicle loading and to passengers carried. 3.2. Category A and B vehicles only : basic knowledge of those items of the vehicle which are vital to the protection of its occupants and to road safety, such as brakes, tyres, oil levels, safety belts, etc.; Category C, D and E vehicles only : knowledge of the function and simple maintenance of the items mentioned above and of all other vehicle parts and devices of particular importance to safety; 3.3. Knowledge of the action which may be required in order to assist road accident victims. PRACTICAL TEST The vehicle and its equipment 4. If a candidate takes the test on a vehicle with automatic transmission, this shall be recorded on any licence issued on the basis of such a test; - Category C vehicles : the permissible maximum weight shall be not less than 7 000 kg; - Category D vehicles : the vehicle shall have not less than 28 seats and shall be not less than 7 m in length; - Category E vehicles : when the towing vehicle belongs to category C, and except in the case of a semi-trailer, the trailer shall have at least two axles, the distance between which shall be greater than 1 m. Contents 5. The principal manoeuvres to be carried out to check the candidate's ability to control the vehicle are as follows: 5.1. Starting on upgrades; 5.2. Category B, C, D and E vehicles only : reversing and reverse turning: 5.3. Braking and stopping at various speeds, including emergency stops if road and traffic conditions permit; 5.4. Category B, C, D and E vehicles only : oblique parking, parking on upgrades and down-grades; 5.5. Turning in a restricted space; 5.6. Category A vehicles only : riding at a slow speed. 6. Behaviour in traffic The main checks to which the candidate will be subjected are: 6.1. Correct positioning on the carriageway; 6.2. Proper negotiation of right and left-hand bends; 6.3. Correct execution of the manoeuvres of changing lanes and turning off at junctions; 6.4. Alertness to other traffic; 6.5. Correct behaviour at intersections, taking due account of all movements of other road users, with special regard to right-of-way; 6.6. Driving at appropriate speeds; 6.7. Use of rear-view mirrors; 6.8. Correct signalling of intended manoeuvres; 6.9. Correct operation of vehicle lighting and warning devices and other ancillary controls; 6.10. Driving with due care and consideration for pedestrians and other road users; 6.11. Correct behaviour with regard to public transport vehicles; 6.12. Compliance with traffic-light signals and instructions given by authorized officials on point duty; 6.13. Appropriate reaction to legally specified signals given by other road users; 6.14. Observance of traffic signs and signals, road markings and pedestrian crossings; 6.15. Observance of appropriate following and lateral distances; 6.16. Correct overtaking; 6.17 Correct use of safety belts if national legislation requires that they be fitted to the vehicle. Sequence of the parts of the test 7. Whenever possible, the part of the test described in paragraph 5 should be carried out before the part described in paragraph 6. Duration of the test 8. The duration of the test and the distance covered shall be sufficient for the checks prescribed in paragraphs 5 and 6 to be carried out. The duration of the part of the test described in paragraph 6 should be more than 30 minutes, but shall not in any case be less than 20 minutes. Location of the test 9. The part of the test described in paragraph 5 may be conducted on a special testing ground, in which case precise criteria should be laid down for measuring objectively the candidate's ability to handle the vehicle. The part of the test described in paragraph 6 shall, wherever possible, be conducted on roads outside built-up areas and on motorways as well as in urban traffic. ANNEX III MINIMUM STANDARDS OF PHYSICAL AND MENTAL FITNESS DEFINITIONS 1. For the purpose of this Annex, drivers are classified into two groups: 1.1. Group 1 : drivers of vehicles of categories A and B; 1.2. Group 2 : drivers of vehicles of categories C, D and E. 2. Similarly, applicants for a first driving licence or for the renewal of a driving licence are classified in the group to which they will belong once the licence has been granted or renewed. MEDICAL EXAMINATIONS 3. Group 1 : applicants shall be required to undergo a medical examination if it becomes apparent, when the necessary formalities are being completed or during the tests which they have to undergo prior to obtaining a driving licence, that they have one or more of the medical disabilities mentioned in this Annex in respect of this group. 4. Group 2 : applicants shall undergo a medical examination before a driving licence is first granted to them and thereafter drivers shall undergo such periodic examinations as may be prescribed by national laws. Eyesight 5. An examination conducted by suitably trained personnel shall be undergone by all applicants for a driving licence. In doubtful cases the applicant shall be referred to a competent medical authority. At the medical examination, attention should be paid to visual acuity, field of vision, night vision, progressive eye diseases, etc. When the wearing of corrective lenses is recognized by the issuing authority as necessary for driving, this shall be recorded on the driving licence. 6. Group 1 : drivers in this group should have their eyesight tested not later than at the age of 70 and preferably earlier, and thereafter at appropriate intervals. If applicants or drivers aged 40 years or more have sub-normal vision after correction but nevertheless meet the minimum requirements given in paragraphs 6.1 and 6.2 below, the cause of loss of vision shall be investigated before driving licences are granted or renewed. Where a disease of the eye is discovered or suspected, the periodic tests should be frequent. 6.1. Applicants for a driving licence or for the renewal of such a licence shall have a visual acuity, with corrective lenses if necessary, of at least 0 74, and preferably of a higher standard in the better eye or of at least 0 75 in both eyes together and, on medical examination, of at least 0 72 in the worse eye. Driving licences shall not be granted or renewed if, on examination, it is shown that there is more than 20 º loss in the temporal part of the applicant's or the driver's field of vision, or if the applicant or driver has diplopia or defective binocular vision. 6.2. Applicants or drivers with sight only in one eye may obtain a driving licence or the renewal of such a licence if the monocular vision is certified by a competent medical authority as having existed for sufficient time to allow adaptation and the visual acuity, with corrective lenses if necessary, is at least 0 78. Such persons must have unrestricted field of vision in their good eye. 7. Group 2 : applicants or drivers in this group shall have their eyesight tested on application for a driving licence and preferably periodically thereafter. If applicants or drivers aged 40 years or more have sub-normal vision after correction but nevertheless meet the minimum requirements given in paragraph 7.1 below, the cause of visual loss shall be investigated before driving licences are granted or renewed. 7.1. Applicants for a driving licence or for the renewal of such a licence must have binocular vision with a visual acuity, with corrective lenses if necessary, of at least 0 775 in the better eye and of at least 0 75 in the worse eye. If corrective lenses are used, the uncorrected vision must not be less than 0 71 and the correction must be tolerated. Driving licences shall not be granted or renewed if the applicant or driver has a restricted field of vision or if he has diplopia or defective binocular vision. 7.2. The use of contact lenses by drivers in this group may be permitted if approved by a competent medical authority. Hearing 8. Driving licences shall not be granted or renewed for applicants or drivers in group 2 if their hearing is so bad that it interferes with the proper discharge of their duties. General physique and physical disabilities 9. Group 1 : unrestricted driving licences shall not be granted or renewed for physically disabled applicants or drivers, unless a driving test has established their ability to operate vehicles with conventional controls. 9.1. Restricted driving licences may be granted or renewed for physically disabled applicants or drivers if the vehicles they drive are adapted to suit the requirements of their disablement. Any restriction on the driving licence shall state the adaptation required on the vehicle. 9.2. In cases of doubt, a practical test shall be made of driving abilities after medical examination by a competent authority and, where appropriate, a driving licence for a limited duration may be issued so as to keep a case under observation. The assessment of physical disablement shall primarily be based on mechanical considerations which make it possible to ascertain whether the disablement is likely to interfere for prolonged periods with efficient and rapid manoeuvring and the handling of controls under all driving conditions, especially in an emergency. 10. Group 2 : driving licences shall not be granted or renewed for applicants or drivers who have any disablement which is likely to prevent proper and safe control of a vehicle. 10.1. Medical examination of applicants or drivers shall cover the full range of body movements - strength, control and coordination - and, in particular, movements of the upper and lower limbs. 10.2. If disablement which is likely to hinder proper and safe control of a vehicle occurs after a driving licence has been granted, the disabled person must give up driving and undergo an examination by a competent medical authority. Cardiovascular diseases 11. Driving licences shall not be granted or renewed for applicants or drivers with cardiovascular diseases unless their request is supported by authorized medical opinion. 12. With regard to applicants or drivers in group 2, the competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. Endocrine disorders 13. In cases of severe endocrine disorders other than diabetes, appropriate provisions in respect of the granting or renewal of driving licences shall be established by the laws of the Member States. 14. Group 1 : driving licences shall not be granted or renewed for applicants or drivers suffering from diabetes who are affected by ocular, nervous or cardiovascular complications or uncompensated acidosis. 14.1. Driving licences may be granted or renewed for a restricted period for applicants or drivers suffering from diabetes who are not affected by any of the complications mentioned in paragraph 14 above, subject to their remaining under authorized medical supervision. 15. Group 2 : driving licences shall not be granted or renewed for applicants or drivers who are diabetics needing insulin treatment. Diseases of the nervous system 16. Driving licences shall not be granted or renewed for applicants or drivers suffering from (a) encephalitis, multiple sclerosis, myasthenia gravis or hereditary diseases of the nervous system associated with progressive muscular atrophy and congenital myotonic disorders; (b) diseases of the peripheral nervous system ; or (c) trauma of the central or peripheral nervous system, unless their application is supported by authorized medical opinion and they are able to handle the controls of a vehicle safely and to comply with traffic regulations. Such cases shall be reviewed at regular intervals. 17. Group 1 : driving licences shall not be granted or renewed for applicants or drivers suffering from epilepsy. National legislation may provide that, subject to authorized medical opinion, licences be granted to persons who have suffered from epilepsy in the past but who have been free from attacks for a long time (e.g. two years). 17.1. Driving licences shall not be granted or renewed for applicants or drivers suffering from cerebrovascular diseases, unless their application is supported by authorized medical opinion and provided that, where necessary, the controls of the vehicle they drive are suitably re-arranged or modified, or that suitable special types of vehicles are used. The duration of the validity of driving licences granted or renewed in such cases shall be limited in accordance with authorized medical opinion. 17.2. Driving licences shall not be granted or renewed for applicants or drivers who have suffered a lesion with damage to the spinal cord and resultant paraplegia unless the vehicle they drive is fitted with special controls. 18. Group 2 : driving licences shall not be granted or renewed for applicants or drivers who suffer or have suffered in the past from epilepsy, a cerebrovascular disease or a lesion with damage to the spinal cord and resulting paraplegia. Mental disorders 19. Driving licences shall not be granted or renewed for applicants or drivers who: (a) suffer from mental disturbance due to disease or trauma of, or operations upon, the central nervous system; (b) suffer from severe mental retardation; (c) suffer from psychosis, which in particular has caused general paralysis ; or (d) suffer from psychoneurosis or personality disorders. unless their application is supported by authorized medical opinion. 20. With regard to applicants or drivers in group 2, the authorized medical authority shall give due consideration to the additional risks and dangers involved in driving the vehicles covered by this group. Alcohol 21. Driving licences shall not be granted or renewed for applicants or drivers who suffer from chronic alcoholism. If the application is supported by an authorized medical opinion, driving licences may be granted or renewed for a limited period for applicants or drivers who suffered from chronic alcoholism in the past. Such cases shall be reviewed at regular intervals. 22. With regard to applicants or drivers in group 2, the authorized medical authority shall give due consideration to the additional risks and dangers involved in driving the vehicles covered by this group. Drugs and medicaments 23. Drug abuse : driving licences shall not be granted or renewed for applicants or drivers who are dependant on psycho-active drugs. 24. Drugs or medicaments taken on a regular basis : driving licences shall not be granted or renewed for applicants or drivers who regularly take drugs or medicaments which can hamper the ability to drive safely, unless their application is supported by authorized medical opinion. 24.1. With regard to applicants or drivers in group 2, the authorized medical authority shall give due consideration to the additional risks and dangers involved in driving the vehicles covered by this group. Diseases of the blood 25. Driving licences shall not be granted or renewed for applicants or drivers suffering from serious diseases of the blood unless the application is supported by authorized medical opinion. Diseases of the genito-urinary system 26. Driving licences shall not be granted or renewed for applicants or drivers suffering from severe renal deficiency. WITHDRAWAL OF DRIVING LICENCES 27. National laws shall include provisions to the effect that, subject to authorized medical opinion, a driving licence shall be withdrawn where the authorities concerned have become aware that the holder's state of health is such that his application for a licence or for its renewal would have been refused. OTHER PROVISIONS (i) The provisions of the Annex shall not prevent a Member State from providing that a driver who has obtained a driving licence before 1 January 1983 under less stringent conditions than those provided for herein may have this licence regularly renewed under the conditions pertaining when he obtained it. (ii) Member States may derogate from the provisions of the Annex where the development of medical science makes such derogations fully compatible with the standards laid down herein. These derogations shall apply only to applicants who have undergone a medical examination and whose application is supported by authorized medical opinion.
============================== "END OF DOC" ==============================
        

        doc 1 :

        'celex': 32009R1072
        'status': In Force
        'act_type': Regulation
        'treaty': TEC (1992)

        full_doc :

        14.11.2009 EN Official Journal of the European Union L 300/72 REGULATION (EC) No 1072/2009 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 21 October 2009 on common rules for access to the international road haulage market (recast) (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty establishing the European Community, and in particular Article 71 thereof, Having regard to the proposal from the Commission, Having regard to the opinion of the European Economic and Social Committee (1), After consulting the Committee of the Regions, Acting in accordance with the procedure laid down in Article 251 of the Treaty (2), Whereas: (1) A number of substantial changes are to be made to Council Regulation (EEC) No 881/92 of 26 March 1992 on access to the market in the carriage of goods by road within the Community to or from the territory of a Member State or passing across the territory of one or more Member States (3), to Council Regulation (EEC) No 3118/93 of 25 October 1993 laying down the conditions under which non-resident carriers may operate national road haulage services within a Member State (4), and to Directive 2006/94/EC of the European Parliament and of the Council of 12 December 2006 on the establishment of common rules for certain types of carriage of goods by road (5). In the interests of clarity and simplification, those legal acts should be recast and incorporated into one single regulation. (2) The establishment of a common transport policy entails, inter alia, laying down common rules applicable to access to the market in the international carriage of goods by road within the territory of the Community, as well as laying down the conditions under which non-resident hauliers may operate transport services within a Member State. Those rules must be laid down in such a way as to contribute to the smooth operation of the internal transport market. (3) To ensure a coherent framework for international road haulage throughout the Community, this Regulation should apply to all international carriage on Community territory. Carriage from Member States to third countries is still largely covered by bilateral agreements between the Member States and those third countries. Therefore, this Regulation should not apply to that part of the journey within the territory of the Member State of loading or unloading as long as the necessary agreements between the Community and the third countries concerned have not been concluded. It should, however, apply to the territory of a Member State crossed in transit. (4) The establishment of a common transport policy implies the removal of all restrictions against the person providing transport services on the grounds of nationality or the fact that he is established in a different Member State from the one in which the services are to be provided. (5) In order to achieve this smoothly and flexibly, provision should be made for a transitional cabotage regime as long as harmonisation of the road haulage market has not yet been completed. (6) The gradual completion of the single European market should lead to the elimination of restrictions on access to the domestic markets of Member States. Nevertheless, this should take into account the effectiveness of controls and the evolution of employment conditions in the profession, the harmonisation of the rules in the fields of, inter alia, enforcement and road user charges, and social and safety legislation. The Commission should closely monitor the market situation as well as the harmonisation mentioned above and propose, if appropriate, the further opening of domestic road transport markets, including cabotage. (7) Under Directive 2006/94/EC, a certain number of types of carriage are exempt from Community authorisation and from any other carriage authorisation. Within the framework of the organisation of the market provided for by this Regulation, a system of exemption from the Community licence and from any other carriage authorisation should be maintained for some of those types of carriage, because of their special nature. (8) Under Directive 2006/94/EC, the carriage of goods with vehicles of a maximum laden weight of between 3,5 tonnes and 6 tonnes was exempt from the requirement for a Community licence. Community rules in the field of road transport of goods, however, apply in general to vehicles with a maximum laden mass of more than 3,5 tonnes. Thus, the provisions of this Regulation should be aligned with the general scope of application of Community road transport rules and should only provide for an exemption for vehicles with a maximum laden mass of up to 3,5 tonnes. (9) The international carriage of goods by road should be conditional on the possession of a Community licence. Hauliers should be required to carry a certified true copy of the Community licence aboard each of their vehicles in order to facilitate effective controls by enforcement authorities, especially those outside the Member State in which the haulier is established. To this end, it is necessary to lay down more detailed specifications as regards the layout and other features of the Community licence and the certified copies. (10) Roadside checks should be carried out without direct or indirect discrimination on grounds of the nationality of the road transport operator or the country of establishment of the road transport operator or of registration of the vehicle. (11) The conditions governing the issue and withdrawal of Community licences and the types of carriage to which they apply, their periods of validity and the detailed rules for their use should be determined. (12) A driver attestation should also be established in order to allow Member States to check effectively whether drivers from third countries are lawfully employed or at the disposal of the haulier responsible for a given transport operation. (13) Hauliers who are holders of Community licences provided for in this Regulation and hauliers authorised to operate certain categories of international haulage service should be permitted to carry out national transport services within a Member State on a temporary basis in conformity with this Regulation, without having a registered office or other establishment therein. When such cabotage operations are performed, they should be subject to Community legislation such as Regulation (EC) No 561/2006 of the European Parliament and of the Council of 15 March 2006 on the harmonisation of certain social legislation relating to road transport (6) and to national law in force in specified areas in the host Member State. (14) Provisions should be adopted to allow action to be taken in the event of serious disturbance of the transport markets affected. For that purpose it is necessary to introduce a suitable decision-making procedure and for the required statistical data to be collected. (15) Without prejudice to the provisions of the Treaty on the right of establishment, cabotage operations consist of the provision of services by hauliers within a Member State in which they are not established and should not be prohibited as long as they are not carried out in a way that creates a permanent or continuous activity within that Member State. To assist the enforcement of this requirement, the frequency of cabotage operations and the period in which they can be performed should be more clearly defined. In the past, such national transport services were permitted on a temporary basis. In practice, it has been difficult to ascertain which services are permitted. Clear and easily enforceable rules are thus needed. (16) This Regulation is without prejudice to the provisions concerning the incoming or outgoing carriage of goods by road as one leg of a combined transport journey as laid down in Council Directive 92/106/EEC of 7 December 1992 on the establishment of common rules for certain types of combined transport of goods between Member States (7). National journeys by road within a host Member State which are not part of a combined transport operation as laid down in Directive 92/106/EEC fall within the definition of cabotage operations and should accordingly be subject to the requirements of this Regulation. (17) The provisions of Directive 96/71/EC of the European Parliament and of the Council of 16 December 1996 concerning the posting of workers in the framework of the provision of services (8) apply to transport undertakings performing a cabotage operation. (18) In order to perform efficient controls of cabotage operations, the enforcement authorities of the host Member States should, at least, have access to data from consignment notes and from recording equipment, in accordance with Council Regulation (EEC) No 3821/85 of 20 December 1985 on recording equipment in road transport (9). (19) Member States should grant each other mutual assistance with a view to the sound application of this Regulation. (20) Administrative formalities should be reduced as far as possible without abandoning the controls and penalties that guarantee the correct application and effective enforcement of this Regulation. To this end, the existing rules on the withdrawal of the Community licence should be clarified and strengthened. The current rules should be adapted to allow the effective sanctioning of serious infringements committed in a host Member State. Penalties should be non-discriminatory and proportionate to the seriousness of the infringements. It should be possible to lodge an appeal in respect of any penalties imposed. (21) Member States should enter in their national electronic register of road transport undertakings all serious infringements committed by hauliers which have led to the imposition of a penalty. (22) In order to facilitate and strengthen the exchange of information between national authorities, Member States should exchange the relevant information through the national contact points set up pursuant to Regulation (EC) No 1071/2009 of the European Parliament and of the Council of 21 October 2009 establishing common rules concerning the conditions to be complied with to pursue the occupation of road transport operator (10). (23) The measures necessary for the implementation of this Regulation should be adopted in accordance with Council Decision 1999/468/EC of 28 June 1999 laying down the procedures for the exercise of implementing powers conferred on the Commission (11). (24) In particular, the Commission should be empowered to adapt Annexes I, II and III to this Regulation to technical progress. Since those measures are of general scope and are designed to amend non-essential elements of this Regulation, they must be adopted in accordance with the regulatory procedure with scrutiny provided for in Article 5a of Decision 1999/468/EC. (25) Member States should take the necessary measures to implement this Regulation, in particular as regards effective, proportionate and dissuasive penalties. (26) Since the objective of this Regulation, namely to ensure a coherent framework for international road haulage throughout the Community, cannot be sufficiently achieved by the Member States and can therefore, by reason of its scale and effects, be better achieved at Community level, the Community may adopt measures, in accordance with the principle of subsidiarity as set out in Article 5 of the Treaty. In accordance with the principle of proportionality, as set out in that Article, this Regulation does not go beyond what is necessary in order to achieve that objective, HAVE ADOPTED THIS REGULATION: CHAPTER I GENERAL PROVISIONS Article 1 Scope 1. This Regulation shall apply to the international carriage of goods by road for hire or reward for journeys carried out within the territory of the Community. 2. In the event of carriage from a Member State to a third country and vice versa, this Regulation shall apply to the part of the journey on the territory of any Member State crossed in transit. It shall not apply to that part of the journey on the territory of the Member State of loading or unloading, as long as the necessary agreement between the Community and the third country concerned has not been concluded. 3. Pending the conclusion of the agreements referred to in paragraph 2, this Regulation shall not affect: (a) provisions relating to the carriage from a Member State to a third country and vice versa included in bilateral agreements concluded by Member States with those third countries; (b) provisions relating to the carriage from a Member State to a third country and vice versa included in bilateral agreements concluded between Member States which, under either bilateral authorisations or liberalisation arrangements, allow loading and unloading in a Member State by hauliers not established in that Member State. 4. This Regulation shall apply to the national carriage of goods by road undertaken on a temporary basis by a non-resident haulier as provided for in Chapter III. 5. The following types of carriage and unladen journeys made in conjunction with such carriage shall not require a Community licence and shall be exempt from any carriage authorisation: (a) carriage of mail as a universal service; (b) carriage of vehicles which have suffered damage or breakdown; (c) carriage of goods in motor vehicles the permissible laden mass of which, including that of trailers, does not exceed 3,5 tonnes; (d) carriage of goods in motor vehicles provided the following conditions are fulfilled: (i) the goods carried are the property of the undertaking or have been sold, bought, let out on hire or hired, produced, extracted, processed or repaired by the undertaking; (ii) the purpose of the journey is to carry the goods to or from the undertaking or to move them, either inside or outside the undertaking for its own requirements; (iii) motor vehicles used for such carriage are driven by personnel employed by, or put at the disposal of, the undertaking under a contractual obligation; (iv) the vehicles carrying the goods are owned by the undertaking, have been bought by it on deferred terms or have been hired provided that in the latter case they meet the conditions of Directive 2006/1/EC of the European Parliament and of the Council of 18 January 2006 on the use of vehicles hired without drivers for the carriage of goods by road (12); and (v) such carriage is no more than ancillary to the overall activities of the undertaking; (e) carriage of medicinal products, appliances, equipment and other articles required for medical care in emergency relief, in particular for natural disasters. Point (d)(iv) of the first subparagraph shall not apply to the use of a replacement vehicle during a short breakdown of the vehicle normally used. 6. The provisions of paragraph 5 shall not affect the conditions under which a Member State authorises its nationals to engage in the activities referred to in that paragraph. Article 2 Definitions For the purposes of this Regulation: 1. vehicle means a motor vehicle registered in a Member State, or a coupled combination of vehicles the motor vehicle of which at least is registered in a Member State, used exclusively for the carriage of goods; 2. international carriage means: (a) a laden journey undertaken by a vehicle the point of departure and the point of arrival of which are in two different Member States, with or without transit through one or more Member States or third countries; (b) a laden journey undertaken by a vehicle from a Member State to a third country or vice versa, with or without transit through one or more Member States or third countries; (c) a laden journey undertaken by a vehicle between third countries, with transit through the territory of one or more Member States; or (d) an unladen journey in conjunction with the carriage referred to in points (a), (b) and (c); 3. host Member State means a Member State in which a haulier operates other than the hauliers Member State of establishment; 4. non-resident haulier means a road haulage undertaking which operates in a host Member State; 5. driver means any person who drives the vehicle even for a short period, or who is carried in a vehicle as part of his duties to be available for driving if necessary; 6. cabotage operations means national carriage for hire or reward carried out on a temporary basis in a host Member State, in conformity with this Regulation; 7. serious infringement of Community road transport legislation means an infringement which may lead to the loss of good repute in accordance with Article 6(1) and (2) of Regulation (EC) No 1071/2009 and/or to the temporary or permanent withdrawal of a Community licence. CHAPTER II INTERNATIONAL CARRIAGE Article 3 General principle International carriage shall be carried out subject to possession of a Community licence and, if the driver is a national of a third country, in conjunction with a driver attestation. Article 4 Community licence 1. The Community licence shall be issued by a Member State, in accordance with this Regulation, to any haulier carrying goods by road for hire or reward who: (a) is established in that Member State in accordance with Community legislation and the national legislation of that Member State; and (b) is entitled in the Member State of establishment, in accordance with Community legislation and the national legislation of that Member State concerning admission to the occupation of road haulage operator, to carry out the international carriage of goods by road. 2. The Community licence shall be issued by the competent authorities of the Member State of establishment for renewable periods of up to 10 years. Community licences and certified copies issued before the date of application of this Regulation shall remain valid until their date of expiry. The Commission shall adapt the period of validity of the Community licence to technical progress, in particular the national electronic registers of road transport undertakings as provided for in Article 16 of Regulation (EC) No 1071/2009. Those measures, designed to amend non-essential elements of this Regulation, shall be adopted in accordance with the regulatory procedure with scrutiny referred to in Article 15(2). 3. The Member State of establishment shall issue the holder with the original of the Community licence, which shall be kept by the haulier, and the number of certified true copies corresponding to the number of vehicles at the disposal of the holder of the Community licence, whether those vehicles are wholly owned or, for example, held under a hire purchase, hire or leasing contract. 4. The Community licence and the certified true copies shall correspond to the model set out in Annex II, which also lays down the conditions governing its use. They shall contain at least two of the security features listed in Annex I. The Commission shall adapt Annexes I and II to technical progress. Those measures, designed to amend non-essential elements of this Regulation, shall be adopted in accordance with the regulatory procedure with scrutiny referred to in Article 15(2). 5. The Community licence and the certified true copies thereof shall bear the seal of the issuing authority as well as a signature and a serial number. The serial numbers of the Community licence and of the certified true copies shall be recorded in the national electronic register of road transport undertakings as part of the data relating to the haulier. 6. The Community licence shall be issued in the name of the haulier and shall be non-transferable. A certified true copy of the Community licence shall be kept in each of the hauliers vehicles and shall be presented at the request of any authorised inspecting officer. In the case of a coupled combination of vehicles, the certified true copy shall accompany the motor vehicle. It shall cover the coupled combination of vehicles even where the trailer or semi-trailer is not registered or authorised to use the roads in the name of the licence holder or where it is registered or authorised to use the roads in another State. Article 5 Driver attestation 1. A driver attestation shall be issued by a Member State, in accordance with this Regulation, to any haulier who: (a) is the holder of a Community licence; and (b) in that Member State, either lawfully employs a driver who is neither a national of a Member State nor a long-term resident within the meaning of Council Directive 2003/109/EC of 25 November 2003 concerning the status of third-country nationals who are long-term residents (13), or lawfully uses a driver who is neither a national of a Member State nor a long-term resident within the meaning of that Directive and who is put at the disposal of that haulier in accordance with the conditions of employment and of vocational training laid down in that Member State: (i) by laws, regulations or administrative provisions; and, as appropriate; (ii) by collective agreements, in accordance with the rules applicable in that Member State. 2. The driver attestation shall be issued by the competent authorities of the Member State of establishment of the haulier, at the request of the holder of the Community licence, for each driver who is neither a national of a Member State nor a long-term resident within the meaning of Directive 2003/109/EC whom that haulier lawfully employs, or for each driver who is neither a national of a Member State nor a long-term resident within the meaning of that Directive and who is put at the disposal of the haulier. Each driver attestation shall certify that the driver named therein is employed in accordance with the conditions laid down in paragraph 1. 3. The driver attestation shall correspond to the model set out in Annex III. It shall contain at least two of the security features listed in Annex I. 4. The Commission shall adapt Annex III to technical progress. Those measures, designed to amend non-essential elements of this Regulation, shall be adopted in accordance with the regulatory procedure with scrutiny referred to in Article 15(2). 5. The driver attestation shall bear the seal of the issuing authority as well as a signature and a serial number. The serial number of the driver attestation may be recorded in the national electronic register of road transport undertakings as part of the data relating to the haulier who puts it at the disposal of the driver designated therein. 6. The driver attestation shall belong to the haulier, who puts it at the disposal of the driver designated therein when that driver drives a vehicle using a Community licence issued to that haulier. A certified true copy of the driver attestation issued by the competent authorities of the hauliers Member State of establishment shall be kept at the hauliers premises. The driver attestation shall be presented at the request of any authorised inspecting officer. 7. A driver attestation shall be issued for a period to be determined by the issuing Member State, subject to a maximum validity of 5 years. Driver attestations issued before the date of application of this Regulation shall remain valid until their date of expiry. The driver attestation shall be valid only as long as the conditions under which it was issued are satisfied. Member States shall take appropriate measures to ensure that if those conditions are no longer met, the haulier returns the attestation immediately to the issuing authorities. Article 6 Verification of conditions 1. Whenever an application for a Community licence or an application for renewal of a Community licence in accordance with Article 4(2) is lodged, the competent authorities of the Member State of establishment shall verify whether the haulier satisfies or continues to satisfy the conditions laid down in Article 4(1). 2. The competent authorities of the Member State of establishment shall regularly verify, by carrying out checks each year covering at least 20 % of the valid driver attestations issued in that Member State, whether the conditions, referred to in Article 5(1), under which a driver attestation has been issued are still satisfied. Article 7 Refusal to issue and withdrawal of Community licence and driver attestation 1. If the conditions laid down in Article 4(1) or those referred to in Article 5(1) are not satisfied, the competent authorities of the Member State of establishment shall reject an application for the issue or renewal of a Community licence or the issue of a driver attestation, by means of a reasoned decision. 2. The competent authorities shall withdraw a Community licence or a driver attestation where the holder: (a) no longer satisfies the conditions laid down in Article 4(1) or those referred to in Article 5(1); or (b) has supplied incorrect information in relation to an application for a Community licence or for a driver attestation. CHAPTER III CABOTAGE Article 8 General principle 1. Any haulier for hire or reward who is a holder of a Community licence and whose driver, if he is a national of a third country, holds a driver attestation, shall be entitled, under the conditions laid down in this Chapter, to carry out cabotage operations. 2. Once the goods carried in the course of an incoming international carriage have been delivered, hauliers referred to in paragraph 1 shall be permitted to carry out, with the same vehicle, or, in the case of a coupled combination, the motor vehicle of that same vehicle, up to three cabotage operations following the international carriage from another Member State or from a third country to the host Member State. The last unloading in the course of a cabotage operation before leaving the host Member State shall take place within 7 days from the last unloading in the host Member State in the course of the incoming international carriage. Within the time limit referred to in the first subparagraph, hauliers may carry out some or all of the cabotage operations permitted under that subparagraph in any Member State under the condition that they are limited to one cabotage operation per Member State within 3 days of the unladen entry into the territory of that Member State. 3. National road haulage services carried out in the host Member State by a non-resident haulier shall only be deemed to conform with this Regulation if the haulier can produce clear evidence of the incoming international carriage and of each consecutive cabotage operation carried out. Evidence referred to in the first subparagraph shall comprise the following details for each operation: (a) the name, address and signature of the sender; (b) the name, address and signature of the haulier; (c) the name and address of the consignee as well as his signature and the date of delivery once the goods have been delivered; (d) the place and the date of taking over of the goods and the place designated for delivery; (e) the description in common use of the nature of the goods and the method of packing, and, in the case of dangerous goods, their generally recognised description, as well as the number of packages and their special marks and numbers; (f) the gross mass of the goods or their quantity otherwise expressed; (g) the number plates of the motor vehicle and trailer. 4. No additional document shall be required in order to prove that the conditions laid down in this Article have been met. 5. Any haulier entitled in the Member State of establishment, in accordance with that Member States legislation, to carry out the road haulage operations for hire or reward specified in Article 1(5)(a), (b) and (c) shall be permitted, under the conditions set out in this Chapter, to carry out, as the case may be, cabotage operations of the same kind or cabotage operations with vehicles in the same category. 6. Permission to carry out cabotage operations, within the framework of the types of carriage referred to in Article 1(5)(d) and (e), shall be unrestricted. Article 9 Rules applicable to cabotage operations 1. The performance of cabotage operations shall be subject, save as otherwise provided in Community legislation, to the laws, regulations and administrative provisions in force in the host Member State with regard to the following: (a) the conditions governing the transport contract; (b) the weights and dimensions of road vehicles; (c) the requirements relating to the carriage of certain categories of goods, in particular dangerous goods, perishable foodstuffs and live animals; (d) the driving time and rest periods; (e) the value added tax (VAT) on transport services. The weights and dimensions referred to in point (b) of the first subparagraph may, where appropriate, exceed those applicable in the hauliers Member State of establishment, but they may under no circumstances exceed the limits set by the host Member State for national traffic or the technical characteristics mentioned in the proofs referred to in Article 6(1) of Council Directive 96/53/EC of 25 July 1996 laying down for certain road vehicles circulating within the Community the maximum authorised dimensions in national and international traffic and the maximum authorised weights in international traffic (14). 2. The laws, regulations and administrative provisions referred to in paragraph 1 shall be applied to non-resident hauliers under the same conditions as those imposed on hauliers established in the host Member State, so as to prevent any discrimination on grounds of nationality or place of establishment. Article 10 Safeguard procedure 1. In the event of serious disturbance of the national transport market in a given geographical area due to, or aggravated by, cabotage, any Member State may refer the matter to the Commission with a view to the adoption of safeguard measures and shall provide the Commission with the necessary information and notify it of the measures it intends to take as regards resident hauliers. 2. For the purposes of paragraph 1: serious disturbance of the national transport market in a given geographical area means the existence on the market of problems specific to it, such that there is a serious and potentially enduring excess of supply over demand, implying a threat to the financial stability and survival of a significant number of hauliers, geographical area means an area covering all or part of the territory of a Member State or extending to all or part of the territory of other Member States. 3. The Commission shall examine the situation on the basis in particular of the relevant data and, after consulting the committee referred to in Article 15(1), shall decide within 1 month of receipt of the Member States request whether or not safeguard measures are necessary and shall adopt them if they are necessary. Such measures may involve the temporary exclusion of the area concerned from the scope of this Regulation. Measures adopted in accordance with this Article shall remain in force for a period not exceeding 6 months, renewable once within the same limits of validity. The Commission shall without delay notify the Member States and the Council of any decision taken pursuant to this paragraph. 4. If the Commission decides to adopt safeguard measures concerning one or more Member States, the competent authorities of the Member States involved shall be required to take measures of equivalent scope in respect of resident hauliers and shall inform the Commission thereof. Those measures shall be applied at the latest as from the same date as the safeguard measures adopted by the Commission. 5. Any Member State may refer to the Council a decision taken by the Commission pursuant to paragraph 3 within 30 days of its notification. The Council, acting by a qualified majority may, within 30 days of that referral, or, if there are referrals by several Member States, of the first referral, take a different decision. The limits of validity laid down in the third subparagraph of paragraph 3 shall apply to the Councils decision. The competent authorities of the Member States concerned shall be required to take measures of equivalent scope in respect of resident hauliers, and shall inform the Commission thereof. If the Council takes no decision within the period referred to in the first subparagraph, the Commission decision shall become final. 6. Where the Commission considers that the measures referred to in paragraph 3 need to be prolonged, it shall submit a proposal to the Council, which shall take a decision by qualified majority. CHAPTER IV MUTUAL ASSISTANCE AND PENALTIES Article 11 Mutual assistance Member States shall assist one another in ensuring the application and monitoring of this Regulation. They shall exchange information via the national contact points established pursuant to Article 18 of Regulation (EC) No 1071/2009. Article 12 Sanctioning of infringements by the Member State of establishment 1. In the event of a serious infringement of Community road transport legislation committed or ascertained in any Member State, the competent authorities of the Member State of establishment of the haulier who has committed such infringement shall take the appropriate action which may include a warning, if provided for by national law, to pursue the matter which may lead, inter alia, to the imposition of the following administrative penalties: (a) temporary or permanent withdrawal of some or all of the certified true copies of the Community licence; (b) temporary or permanent withdrawal of the Community licence. These penalties may be determined after the final decision on the matter has been taken and shall have regard to the seriousness of the infringement committed by the holder of the Community licence and to the total number of certified true copies of that licence that he holds in respect of international traffic. 2. In the event of a serious infringement regarding any misuse whatsoever of driver attestations, the competent authorities of the Member State of establishment of the haulier who committed such infringement shall impose appropriate penalties, such as: (a) suspending the issue of driver attestations; (b) withdrawing driver attestations; (c) making the issue of driver attestations subject to additional conditions in order to prevent misuse; (d) withdrawing, temporarily or permanently, some or all of the certified true copies of the Community licence; (e) withdrawing, temporarily or permanently, the Community licence. These penalties may be determined after the final decision on the matter has been taken and shall have regard to the seriousness of the infringement committed by the holder of the Community licence. 3. The competent authorities of the Member State of establishment shall communicate to the competent authorities of the Member State in which the infringement was ascertained, as soon as possible and at the latest within 6 weeks of their final decision on the matter, which, if any, of the penalties provided for in paragraphs 1 and 2 have been imposed. If such penalties are not imposed, the competent authorities of the Member State of establishment shall state the reasons therefor. 4. The competent authorities shall ensure that the penalties imposed on the haulier concerned are, as a whole, proportionate to the infringement or infringements which gave rise to such penalties, taking into account any penalty for the same infringement imposed in the Member State in which the infringement was ascertained. 5. The competent authorities of the hauliers Member State of establishment may also, pursuant to national law, bring proceedings against the haulier before a competent national court or tribunal. They shall inform the competent authority of the host Member State of any decisions taken to this effect. 6. Member States shall ensure that hauliers have the right to appeal against any administrative penalty imposed on them pursuant to this Article. Article 13 Sanctioning of infringements by the host Member State 1. Where the competent authorities of a Member State are aware of a serious infringement of this Regulation or of Community road transport legislation attributable to a non-resident haulier, the Member State within the territory of which the infringement is ascertained shall transmit to the competent authorities of the hauliers Member State of establishment, as soon as possible and at the latest within 6 weeks of their final decision on the matter, the following information: (a) a description of the infringement and the date and time when it was committed; (b) the category, type and seriousness of the infringement; and (c) the penalties imposed and the penalties executed. The competent authorities of the host Member State may request the competent authorities of the Member State of establishment to impose administrative penalties in accordance with Article 12. 2. Without prejudice to any criminal prosecution, the competent authorities of the host Member State shall be empowered to impose penalties on a non-resident haulier who has committed infringements of this Regulation or of national or Community road transport legislation in their territory during a cabotage operation. They shall impose such penalties on a non-discriminatory basis. These penalties may, inter alia, consist of a warning, or, in the event of a serious infringement, a temporary ban on cabotage operations on the territory of the host Member State where the infringement was committed. 3. Member States shall ensure that hauliers have the right to appeal against any administrative penalty imposed on them pursuant to this Article. Article 14 Entry in the national electronic registers Member States shall ensure that serious infringements of Community road transport legislation committed by hauliers established in their territory, which have led to the imposition of a penalty by any Member State, as well as any temporary or permanent withdrawal of the Community licence or of the certified true copy thereof, are recorded in the national electronic register of road transport undertakings. Entries in the register which concern a temporary or permanent withdrawal of a Community licence shall remain in the database for 2 years from the time of the expiry of the period of withdrawal, in the case of temporary withdrawal, or from the date of withdrawal, in the case of permanent withdrawal. CHAPTER V IMPLEMENTATION Article 15 Committee procedure 1. The Commission shall be assisted by the committee established by Article 18(1) of Regulation (EEC) No 3821/85. 2. Where reference is made to this paragraph, Article 5a(1) to (4) and Article 7 of Decision 1999/468/EC shall apply, having regard to the provisions of Article 8 thereof. Article 16 Penalties Member States shall lay down the rules on penalties applicable to infringements of the provisions of this Regulation, and shall take all the measures necessary to ensure that they are implemented. The penalties provided for must be effective, proportionate and dissuasive. Member States shall notify those provisions to the Commission by 4 December 2011, and shall notify it without delay of any subsequent amendment affecting them. Member States shall ensure that all such measures are taken without discrimination as to the nationality or place of establishment of the haulier. Article 17 Reporting 1. Every 2 years Member States shall inform the Commission of the number of hauliers possessing Community licences on 31 December of the previous year and of the number of certified true copies corresponding to the vehicles in circulation at that date. 2. Member States shall also inform the Commission of the number of driver attestations issued in the previous calendar year as well as the number of driver attestations in circulation on 31 December of that same year. 3. The Commission shall draw up a report on the state of the Community road transport market by the end of 2013. The report shall contain an analysis of the market situation, including an evaluation of the effectiveness of controls and the evolution of employment conditions in the profession, as well as an assessment as to whether harmonisation of the rules in the fields, inter alia, of enforcement and road user charges, as well as social and safety legislation, has progressed to such an extent that the further opening of domestic road transport markets, including cabotage, could be envisaged. CHAPTER VI FINAL PROVISIONS Article 18 Repeals Regulations (EEC) No 881/92 and (EEC) No 3118/93 and Directive 2006/94/EC are hereby repealed. References to the repealed Regulations and Directive shall be construed as references to this Regulation and shall be read in accordance with the correlation table set out in Annex IV. Article 19 Entry into force This Regulation shall enter into force on the 20th day following its publication in the Official Journal of the European Union. It shall apply from 4 December 2011, with the exception of Articles 8 and 9, which shall apply from 14 May 2010. This Regulation shall be binding in its entirety and directly applicable in all Member States. Done at Strasbourg, 21 October 2009. For the European Parliament The President J. BUZEK For the Council The President C. MALMSTRÃ M (1) OJ C 204, 9.8.2008, p. 31. (2) Opinion of the European Parliament of 21 May 2008 (not yet published in the Official Journal), Council Common Position of 9 January 2009 (OJ C 62 E, 17.3.2009, p. 46), Position of the European Parliament of 23 April 2009 (not yet published in the Official Journal) and Council Decision of 24 September 2009. (3) OJ L 95, 9.4.1992, p. 1. (4) OJ L 279, 12.11.1993, p. 1. (5) OJ L 374, 27.12.2006, p. 5. (6) OJ L 102, 11.4.2006, p. 1. (7) OJ L 368, 17.12.1992, p. 38. (8) OJ L 18, 21.1.1997, p. 1. (9) OJ L 370, 31.12.1985, p. 8. (10) See page 51 of this Official Journal. (11) OJ L 184, 17.7.1999, p. 23. (12) OJ L 33, 4.2.2006, p. 82. (13) OJ L 16, 23.1.2004, p. 44. (14) OJ L 235, 17.9.1996, p. 59. ANNEX I Security features of the Community licence and the driver attestation The Community licence and the driver attestation must have at least two of the following security features:  a hologram,  special fibres in the paper which become visible under UV-light,  at least one microprint line (printing visible only with a magnifying glass and not reproduced by photocopying machines),  tactile characters, symbols or patterns,  double numbering: serial number of the Community licence, of the certified copy thereof or of the driver attestation as well as, in each case, the issue number,  a security design background with fine guilloche patterns and rainbow printing. ANNEX II Community licence model EUROPEAN COMMUNITY (a) (Colour Pantone light blue, format DIN A4 cellulose paper 100 g/m2 or more) (First page of the licence) (Text in (one of) the official language(s) of the Member State issuing the licence) (b) (Second page of the licence) (Text in (one of) the official language(s) of the Member State issuing the licence) GENERAL PROVISIONS This licence is issued under Regulation (EC) No 1072/2009. It entitles the holder to engage in the international carriage of goods by road for hire or reward by any route for journeys or parts of journeys carried out within the territory of the Community and, where appropriate, subject to the conditions laid down herein:  where the point of departure and the point of arrival are situated in two different Member States, with or without transit through one or more Member States or third countries,  from a Member State to a third country or vice versa, with or without transit through one or more Member States or third countries,  between third countries with transit through the territory of one or more Member States, and unladen journeys in connection with such carriage. In the case of carriage from a Member State to a third country or vice versa, this licence is valid for that part of the journey carried out within the territory of the Community. It shall be valid in the Member State of loading or unloading only after the conclusion of the necessary agreement between the Community and the third country in question in accordance with Regulation (EC) No 1072/2009. The licence is personal to the holder and is non-transferable. It may be withdrawn by the competent authority of the Member State which issued it, notably where the holder has:  not complied with all the conditions for using the licence,  supplied incorrect information with regard to the data needed for the issue or extension of the licence. The original of the licence must be kept by the haulage undertaking. A certified copy of the licence must be kept in the vehicle (1). In the case of a coupled combination of vehicles it must accompany the motor vehicle. It covers the coupled combination of vehicles even if the trailer or semi-trailer is not registered or authorised to use the roads in the name of the licence holder or if it is registered or authorised to use the roads in another State. The licence must be presented at the request of any authorised inspecting officer. Within the territory of each Member State, the holder must comply with the laws, regulations and administrative provisions in force in that State, in particular with regard to transport and traffic. (1) Vehicle means a motor vehicle registered in a Member State, or a coupled combination of vehicles the motor vehicle of which at least is registered in a Member State, used exclusively for the carriage of goods. ANNEX III Driver attestation model EUROPEAN COMMUNITY (a) (Colour Pantone pink, format DIN A4 cellulose paper 100g/m2 or more) (First page of the attestation) (Text in (one of) the official language(s) of the Member State issuing the attestation) (b) (Second page of the attestation) (Text in (one of) the official language(s) of the Member State issuing the attestation) GENERAL PROVISIONS This attestation is issued under Regulation (EC) No 1072/2009. It certifies that the driver named therein is employed, in accordance with the laws, regulations or administrative provisions and, as appropriate, the collective agreements, in accordance with the rules applicable in the Member State mentioned on the attestation, on the conditions of employment and of vocational training of drivers applicable in that Member State to carry out road operations in that State. The driver attestation shall belong to the haulier, who puts it at the disposal of the driver designated therein when that driver drives a vehicle (1) engaged in carriage using a Community licence issued to that haulier. The driver attestation is not transferable. The driver attestation shall be valid only as long as the conditions under which it was issued are still satisfied and must be returned immediately by the haulier to the issuing authorities if these conditions are no longer met. It may be withdrawn by the competent authority of the Member State which issued it, in particular where the holder has:  not complied with all the conditions for using the attestation,  supplied incorrect information with regard to the data needed for the issue or extension of the attestation. A certified true copy of the attestation must be kept by the haulage undertaking. An original attestation must be kept in the vehicle and must be presented by the driver at the request of any authorised inspecting officer. (1) Vehicle means a motor vehicle registered in a Member State, or a coupled combination of vehicles the motor vehicle of which at least is registered in a Member State, used exclusively for the carriage of goods. ANNEX IV Correlation Table Regulation (EEC) No 881/92 Regulation (EEC) No 3118/93 Directive 2006/94/EC This Regulation Article 1(1) Article 1(1) Article 1(2) Article 1(2) Article 1(3) Article 1(3) Annex II Article 1(1) and (2), Annex I; Article 2 Article 1(5) Article 2 Article 1(6) Article 2 Article 2 Article 3(1) Article 3 Article 3(2) Article 4(1) Article 3(3) Article 5(1) Article 4 Article 5(1) Article 4(2) Article 5(2) Article 4(3) Article 5(3) Article 4(4) Article 4(5) Article 5(4), Annex I Article 4(6) Article 5(5) Article 4(2) Article 6(1) Article 5(2) Article 6(2) Article 5(2) Article 6(3) Article 5(3) Article 6(4) Article 5(6) Article 6(5) Article 5(7) Article 7 Article 6 Article 8(1) Article 7(1) Article 8(2) Article 7(2) Article 8(3) Article 12(1) Article 8(4) Article 12(2) Article 9(1) and (2) Article 12(6) Article 1(1) Article 8(1) Article 1(2) Article 8(5) Article 1(3) and (4) Article 8(6) Article 2 Article 3 Article 4 Article 5 Article 6(1) Article 9(1) Article 6(2) Article 6(3) Article 9(2) Article 6(4) Article 7 Article 10 Article 10 Article 17(1) Article 11(1) Article 8(1) Article 11 Article 11(2) Article 13(1) Article 11(3) Article 12(4) Article 11a Article 8(2) and (3) Article 13(2) Article 8(4), first and third subparagraphs Article 8(4), second subparagraph Article 12(4) Article 8(4), fourth and fifth subparagraphs Article 12(5) Article 9 Article 13(3) Article 12 Article 18 Article 13 Article 14 Article 10 Article 11 Article 15 Article 12 Article 4 Article 19 Article 3 Article 5 Annex II, III Annex I Annex II Annex III Annex III Annex I Annex II Annex III Annex IV
============================== "END OF DOC" ==============================
        

        doc 2 :

        'celex': 32015L0413
        'status': In Force
        'act_type': Directive
        'treaty': TFEU (2008)

        full_doc :

        13.3.2015 EN Official Journal of the European Union L 68/9 DIRECTIVE (EU) 2015/413 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 11 March 2015 facilitating cross-border exchange of information on road-safety-related traffic offences (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 91(1)(c) thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the European Economic and Social Committee (1), After consulting the Committee of the Regions, Acting in accordance with the ordinary legislative procedure (2), Whereas: (1) Improving road safety is a prime objective of the Union's transport policy. The Union is pursuing a policy to improve road safety with the objective of reducing fatalities, injuries and material damage. An important element of that policy is the consistent enforcement of sanctions for road traffic offences committed in the Union which considerably jeopardise road safety. (2) However, due to a lack of appropriate procedures and notwithstanding existing possibilities under Council Decision 2008/615/JHA (3) and Council Decision 2008/616/JHA (4) (the PrÃ ¼m Decisions), sanctions in the form of financial penalties for certain road traffic offences are often not enforced if those offences are committed with a vehicle which is registered in a Member State other than the Member State where the offence took place. This Directive aims to ensure that even in such cases, the effectiveness of the investigation of road-safety-related traffic offences should be ensured. (3) In its communication of 20 July 2010 entitled Towards a European road safety area: policy orientations on road safety 2011-2020, the Commission emphasised that enforcement of road traffic rules remains a key factor in creating the conditions for a considerable reduction in the number of deaths and injuries. In its conclusions of 2 December 2010 on road safety, the Council called for consideration of the need for further strengthening of enforcement of road traffic rules by Member States and, where appropriate, at Union level. It invited the Commission to examine the possibilities of harmonising traffic rules at Union level where appropriate and adopting further measures on facilitating cross-border enforcement with regard to road traffic offences, in particular those related to serious traffic accidents. (4) On 19 March 2008, the Commission adopted a proposal for a Directive of the European Parliament and of the Council facilitating cross-border enforcement in the field of road safety on the basis of Article 71(1)(c) of the Treaty establishing the European Community (now Article 91 of Treaty on the Functioning of the European Union (TFEU)). Directive 2011/82/EU of the European Parliament and of the Council (5) was, however, adopted on the basis of Article 87(2) TFEU. The judgment of the Court of Justice of 6 May 2014 in Case C-43/12 (6) annulled Directive 2011/82/EU on the grounds that it could not validly be adopted on the basis of Article 87(2) TFEU. The judgment maintained the effects of Directive 2011/82/EU until the entry into force within a reasonable period of time  which is not to exceed 12 months as from the date of delivery of the judgment  of a new directive based on Article 91(1)(c) TFEU. Therefore a new Directive should be adopted on the basis of that Article. (5) Greater convergence of control measures between Member States should be encouraged and the Commission should examine in this respect the need for developing common standards for automatic checking equipment for road safety controls. (6) The awareness of Union citizens should be raised as regards the road safety traffic rules in force in different Member States and as regards the implementation of this Directive, in particular through appropriate measures guaranteeing the provision of sufficient information on the consequences of not respecting the road safety traffic rules when travelling in a Member State other than the Member State of registration. (7) In order to improve road safety throughout the Union and to ensure equal treatment of drivers, namely resident and non-resident offenders, enforcement should be facilitated irrespective of the Member State of registration of the vehicle. To this end, a system of cross-border exchange of information should be used for certain identified road-safety-related traffic offences, regardless of their administrative or criminal nature under the law of the Member State concerned, granting the Member State of the offence access to vehicle registration data (VRD) of the Member State of registration. (8) A more efficient cross-border exchange of VRD, which should facilitate the identification of persons suspected of committing a road-safety-related traffic offence, might increase the deterrent effect and induce more cautious behaviour by the driver of a vehicle that is registered in a Member State other than the Member State of the offence, thereby preventing casualties due to road traffic accidents. (9) The road-safety-related traffic offences covered by this Directive are not subject to homogeneous treatment in the Member States. Some Member States qualify such offences under national law as administrative offences while others qualify them as criminal offences. This Directive should apply regardless of how those offences are qualified under national law. (10) Member States should grant each other the right of access to their VRD in order to improve the exchange of information and to speed up the procedures in force. To this end, the provisions concerning the technical specifications and the availability of automated data exchange set out in the PrÃ ¼m Decisions should, as far as possible, be included in this Directive. (11) Decision 2008/616/JHA specifies the security features for existing software applications and the related technical requirements for the exchange of vehicle registration data. Without prejudice to the general applicability of that Decision, those security features and technical requirements should, for reasons of regulatory and practical efficiency, be used for the purposes of this Directive. (12) Existing software applications should be the basis for the data exchange under this Directive and should, at the same time, also facilitate the reporting by Member States to the Commission. Such applications should provide for the expeditious, secure and confidential exchange of specific VRD between Member States. Advantage should be taken of the European Vehicle and Driving Licence Information System (Eucaris) software application, which is mandatory for Member States under the PrÃ ¼m Decisions as regards VRD. The Commission should assess and report on the functioning of the software applications used for the purposes of this Directive. (13) The scope of those software applications should be limited to the processes used in the exchange of information between the national contact points in the Member States. Procedures and automated processes in which the information is to be used are outside the scope of such applications. (14) The Information Management Strategy for EU internal security aims to find the simplest and most easily traceable and cost-effective solutions for data exchange. (15) Member States should be able to contact the owner, the holder of the vehicle or the otherwise identified person suspected of committing the road-safety-related traffic offence in order to keep the person concerned informed of the applicable procedures and the legal consequences under the law of the Member State of the offence. In doing so, Member States should consider sending the information concerning road-safety-related traffic offences in the language of the registration documents, or in the language most likely to be understood by the person concerned, to ensure that that person has a clear understanding of the information which is being shared with the person concerned. Member States should apply the appropriate procedures to ensure that only the person concerned is informed and not a third party. To that effect, Member States should use detailed arrangements similar to those adopted for following up such offences including means such as, where appropriate, registered delivery. This will allow that person to respond to the information letter in an appropriate way, in particular by asking for more information, by settling the fine or by exercising his/her rights of defence, especially in the case of mistaken identity. Further proceedings are covered by applicable legal instruments, including instruments on mutual assistance and on mutual recognition, for example Council Framework Decision 2005/214/JHA (7). (16) Member States should provide equivalent translation with respect to the information letter sent by the Member State of the offence, as provided for in Directive 2010/64/EU of the European Parliament and of the Council (8). (17) With a view to pursuing a road safety policy that aims to provide a high level of protection for all road users in the Union, and taking into account the widely differing circumstances pertaining within the Union, Member States should act, without prejudice to more restrictive policies and laws, in order to ensure greater convergence of road traffic rules and of their enforcement between Member States. In the framework of its report to the European Parliament and to the Council on the application of this Directive, the Commission should examine the need to develop common standards in order to establish comparable methods, practices and minimum standards at Union level taking into account international cooperation and existing agreements in the field of road safety, in particular the Vienna Convention on Road Traffic of 8 November 1968. (18) In its report to the European Parliament and to the Council on the application of this Directive by the Member States, the Commission should examine the need for common criteria for follow-up procedures by Member States in the event of non-payment of a financial penalty, in accordance with Member States' laws and procedures. In that report, the Commission should address issues such as the procedures between the competent authorities of the Member States for the transmission of the final decision to impose a sanction and/or financial penalty as well as the recognition and enforcement of the final decision. (19) In preparing the review of this Directive, the Commission should consult the relevant stakeholders, such as road safety and law enforcement authorities or competent bodies, victims' associations and other non-governmental organisations active in the field of road safety. (20) Closer cooperation between law enforcement authorities should go hand in hand with respect for fundamental rights, in particular the right to respect for privacy and to the protection of personal data, guaranteed by special data protection arrangements. Those arrangements should take particular account of the specific nature of cross-border online access to databases. It is necessary that the software applications to be set up enable the exchange of information to be carried out in secure conditions and ensure the confidentiality of the data transmitted. The data collected under this Directive should not be used for purposes other than those of this Directive. Member States should comply with the obligations on the conditions of use and of temporary storage of the data. (21) The processing of personal data provided by this Directive is appropriate for attaining the legitimate aims pursued by this Directive in the field of road safety, namely to ensure a high level of protection for all road users in the Union by facilitating the cross-border exchange of information on road-safety-related traffic offences and, thereby, the enforcement of sanctions, and does not exceed what is appropriate and necessary in order to achieve those objectives. (22) Data relating to the identification of an offender are personal data. Directive 95/46/EC of the European Parliament and of the Council (9) should apply to the processing activities carried out in application of this Directive. Without prejudice to the procedural requirements for appeal and the redress mechanisms of the Member State concerned, the data subject should accordingly be informed, when notified of the offence, of the right to access and the right to rectification and deletion of personal data, as well as of the maximum legal storage period of the data. In this context, the data subject should also have the right to obtain the correction of any inaccurate personal data or the immediate deletion of any data recorded unlawfully. (23) In the framework of the PrÃ ¼m Decisions, the processing of VRD containing personal data is subject to the specific provisions on data protection set out in Decision 2008/615/JHA. In that respect, Member States have the possibility to apply those specific provisions to personal data which are also processed for the purposes of this Directive provided that they ensure that the processing of data related to all of the offences covered by this Directive complies with the national provisions implementing Directive 95/46/EC. (24) It should be possible for third countries to participate in the exchange of VRD provided that they have concluded an agreement with the Union to this effect. Such an agreement would have to include necessary provisions on data protection. (25) This Directive upholds the fundamental rights and principles recognised by the Charter of Fundamental Rights of the European Union, including the respect for private and family life, the protection of personal data, the right to a fair trial, the presumption of innocence and the right of defence. (26) In order to achieve the objective of the exchange of information between Member States through interoperable means, the power to adopt acts in accordance with Article 290 TFEU should be delegated to the Commission in respect of the taking into account of relevant changes to PrÃ ¼m Decisions or where required by legal acts of the Union directly relevant for the updating of Annex I. It is of particular importance that the Commission follow its usual practice and carry out appropriate consultations during its preparatory work, including at expert level. The Commission, when preparing and drawing up delegated acts, should ensure a simultaneous, timely and appropriate transmission of relevant documents to the European Parliament and to the Council. (27) The Commission should analyse the application of this Directive with a view to identifying further effective and efficient measures to improve road safety. Without prejudice to obligations to transpose this Directive, Denmark, Ireland and the United Kingdom should also cooperate with the Commission in this work, where appropriate, to ensure timely and complete reporting on this matter. (28) Since the objective of this Directive, namely to ensure a high level of protection for all road users in the Union by facilitating the cross-border exchange of information on road-safety-related traffic offences, where they are committed with a vehicle registered in a Member State other than the Member State where the offence took place, cannot be sufficiently achieved by the Member States, but can rather, by reason of the scale and effects of the action, be better achieved at Union level, the Union may adopt measures, in accordance with the principle of subsidiarity, as set out in Article 5 of the Treaty on European Union. In accordance with the principle of proportionality, as set out in that Article, this Directive does not go beyond what is necessary in order to achieve that objective. (29) Given that Denmark, Ireland and the United Kingdom were not subject to Directive 2011/82/EU and therefore have not transposed it, it is appropriate to allow those Member States sufficient additional time to do so. (30) The European Data Protection Supervisor was consulted in accordance with Article 28(2) of Regulation (EC) No 45/2001 of the European Parliament and of the Council (10) and delivered an opinion on 3 October 2014, HAVE ADOPTED THIS DIRECTIVE: Article 1 Objective This Directive aims to ensure a high level of protection for all road users in the Union by facilitating the cross-border exchange of information on road-safety-related traffic offences, and thereby facilitating the enforcement of sanctions, where those offences are committed with a vehicle registered in a Member State other than the Member State in which the offence took place. Article 2 Scope This Directive applies to the following road-safety-related traffic offences: (a) speeding; (b) failing to use a seat-belt; (c) failing to stop at a red traffic light; (d) drink-driving; (e) driving while under the influence of drugs; (f) failing to wear a safety helmet; (g) the use of a forbidden lane; (h) illegally using a mobile telephone or any other communication devices while driving. Article 3 Definitions For the purposes of this Directive, the following definitions apply: (a) vehicle means any power-driven vehicle, including motorcycles, which is normally used for carrying persons or goods by road; (b) Member State of the offence means the Member State where the offence was committed; (c) Member State of registration means the Member State where the vehicle with which the offence was committed is registered; (d) speeding means exceeding speed limits in force in the Member State of offence for the road or type of vehicle concerned; (e) failing to use a seat-belt means not complying with the requirement to wear a seat-belt or to use a child restraint in accordance with Council Directive 91/671/EEC (11) and the law of the Member State of the offence; (f) failing to stop at a red traffic light means driving through a red traffic light or any other relevant stop signal, as defined in the law of the Member State of the offence; (g) drink-driving means driving while impaired by alcohol, as defined in the law of the Member State of the offence; (h) driving under the influence of drugs means driving while impaired by drugs or other substances having a similar effect, as defined in the law of the Member State of the offence; (i) failing to wear a safety helmet means not wearing a safety helmet, as defined in the law of the Member State of the offence; (j) use of a forbidden lane means illegally using part of a road section, such as an emergency lane, public transport lane or temporary closed lane for reasons of congestion or road works, as defined in the law of the Member State of the offence; (k) illegally using a mobile telephone or any other communication devices while driving means illegally using a mobile telephone or any other communication devices while driving, as defined in the law of the Member State of the offence; (l) national contact point means a designated competent authority for the exchange of VRD; (m) automated search means an online access procedure for consulting the databases of one, more than one, or all of the Member States or of the participating countries; (n) holder of the vehicle means the person in whose name the vehicle is registered, as defined in the law of the Member State of registration. Article 4 Procedure for the exchange of information between Member States 1. For the investigation of the road-safety-related traffic offences referred to in Article 2, the Member State shall grant other Member States' national contact points, referred to in paragraph 2 of this Article, access to the following national VRD, with the power to conduct automated searches thereon: (a) data relating to vehicles; and (b) data relating to owners or holders of the vehicle. The data elements referred to in points (a) and (b) which are necessary to conduct a search shall be in compliance with Annex I. 2. For the purposes of the exchange of data referred to in paragraph 1, each Member State shall designate a national contact point. The powers of the national contact points shall be governed by the applicable law of the Member State concerned. 3. When conducting a search in the form of an outgoing request, the national contact point of the Member State of the offence shall use a full registration number. Those searches shall be conducted in compliance with the procedures as described in Chapter 3 of the Annex to Decision 2008/616/JHA, except for point 1 of Chapter 3 of the Annex to Decision 2008/616/JHA, for which Annex I to this Directive shall apply. The Member State of the offence shall, under this Directive, use the data obtained in order to establish who is personally liable for road-safety-related traffic offences listed in Article 2 of this Directive. 4. Member States shall take all necessary measures to ensure that the exchange of information is carried out by interoperable electronic means without exchange of data involving other databases which are not used for the purposes of this Directive. Member States shall ensure that such exchange of information is conducted in a cost-efficient and secure manner. Member States shall ensure the security and protection of the data transmitted, as far as possible using existing software applications such as the one referred to in Article 15 of Decision 2008/616/JHA and amended versions of those software applications, in compliance with Annex I to this Directive and with points 2 and 3 of Chapter 3 of the Annex to Decision 2008/616/JHA. The amended versions of the software applications shall provide for both online real-time exchange mode and batch exchange mode, the latter allowing for the exchange of multiple requests or responses within one message. 5. Each Member State shall bear its own costs arising from the administration, use and maintenance of the software applications referred to in paragraph 4. Article 5 Information letter on the road-safety-related traffic offences 1. The Member State of the offence shall decide whether or not to initiate follow-up proceedings in relation to the road-safety-related traffic offences listed in Article 2. Where the Member State of the offence decides to initiate such proceedings, that Member State shall, in accordance with its national law, inform the owner, the holder of the vehicle or the otherwise identified person suspected of committing the road-safety-related traffic offence. This information shall, as applicable under national law, include the legal consequences thereof within the territory of the Member State of the offence under the law of that Member State. 2. When sending the information letter to the owner, the holder of the vehicle or to the otherwise identified person suspected of committing the road-safety-related traffic offence, the Member State of the offence shall, in accordance with its law, include any relevant information, notably the nature of this road-safety-related traffic offence, the place, date and time of the offence, the title of the texts of the national law infringed and the sanction and, where appropriate, data concerning the device used for detecting the offence. For that purpose, the Member State of the offence may use the template set out in Annex II. 3. Where the Member State of the offence decides to initiate follow-up proceedings in relation to the road-safety-related traffic offences listed in Article 2, the Member State of the offence, for the purpose of ensuring the respect of fundamental rights, sends the information letter in the language of the registration document of the vehicle, if available, or in one of the official languages of the Member State of registration. Article 6 Reporting by Member States to the Commission Each Member State shall send a comprehensive report to the Commission by 6 May 2016 and every two years thereafter. The comprehensive report shall indicate the number of automated searches conducted by the Member State of the offence addressed to the national contact point of the Member State of registration, following offences committed on its territory, together with the type of offences for which requests were addressed and the number of failed requests. The comprehensive report shall also include a description of the situation at national level in relation to the follow-up given to the road-safety-related traffic offences, based on the proportion of such offences which have been followed up by information letters. Article 7 Data protection 1. The provisions on data protection set out in Directive 95/46/EC shall apply to personal data processed under this Directive. 2. In particular, each Member State shall ensure that personal data processed under this Directive are, within an appropriate time period, rectified if inaccurate, or erased or blocked when they are no longer required, in accordance with Articles 6 and 12 of Directive 95/46/EC, and that a time limit for the storage of data is established in accordance with Article 6 of that Directive. Member States shall ensure that all personal data processed under this Directive are only used for the objective set out in Article 1 of this Directive, and that the data subjects have the same rights to information, to access, to rectification, erasure and blocking, to compensation and to judicial redress as those adopted under national law in implementation of the relevant provisions of Directive 95/46/EC. 3. Any person concerned shall have the right to obtain information on which personal data recorded in the Member State of registration were transmitted to the Member State of the offence, including the date of the request and the competent authority of the Member State of the offence. Article 8 Information for road users in the Union 1. The Commission shall make available on its website a summary in all official languages of the institutions of the Union of the rules in force in Member States in the field covered by this Directive. Member States shall provide information on these rules to the Commission. 2. Member States shall provide road users with the necessary information about the rules applicable in their territory and the measures implementing this Directive in association with, among other organisations, road safety bodies, non-governmental organisations active in the field of road safety and automobile clubs. Article 9 Delegated acts The Commission shall be empowered to adopt delegated acts, in accordance with Article 10, updating Annex I in the light of technical progress to take into account relevant changes to PrÃ ¼m Decisions or where this is required by legal acts of the Union directly relevant to the updating of Annex I. Article 10 Exercise of the delegation 1. The power to adopt delegated acts is conferred on the Commission subject to the conditions laid down in this Article. 2. The power to adopt delegated acts referred to in Article 9 shall be conferred on the Commission for a period of five years from 13 March 2015. The Commission shall draw up a report in respect of the delegation of power not later than nine months before the end of the five-year period. The delegation of power shall be tacitly extended for periods of an identical duration, unless the European Parliament or the Council opposes such extension not later than three months before the end of each period. 3. The delegation of power referred to in Article 9 may be revoked at any time by the European Parliament or by the Council. A decision to revoke shall put an end to the delegation of the power specified in that decision. It shall take effect on the day following the publication of the decision in the Official Journal of the European Union or at a later date specified therein. It shall not affect the validity of any delegated acts already in force. 4. It is of particular importance that the Commission follow its usual practice and carry out consultations with experts, including Member States' experts, before adopting those delegated acts. As soon as it adopts a delegated act, the Commission shall notify it simultaneously to the European Parliament and to the Council. 5. A delegated act adopted pursuant to Article 9 shall enter into force only if no objection has been expressed either by the European Parliament or the Council within a period of two months of notification of that act to the European Parliament and the Council or if, before the expiry of that period, the European Parliament and the Council have both informed the Commission that they will not object. That period shall be extended by two months at the initiative of the European Parliament or of the Council. Article 11 Revision of the Directive Without prejudice to the provisions laid down in the second subparagraph of Article 12(1), the Commission shall, by 7 November 2016, submit a report to the European Parliament and to the Council on the application of this Directive by the Member States. In its report, the Commission shall focus in particular on, and shall, as appropriate, make proposals to cover, the following aspects:  an assessment of whether other road-safety-related traffic offences should be added to the scope of this Directive,  an assessment of the effectiveness of this Directive on the reduction in the number of fatalities on Union roads,  an assessment of the need for developing common standards for automatic checking equipment and for procedures. In this context, the Commission is invited to develop at Union level road safety guidelines within the framework of the common transport policy in order to ensure greater convergence of the enforcement of road traffic rules by Member States through comparable methods and practices. These guidelines may cover at least the offences listed in points (a) to (d) of Article 2,  an assessment of the need to strengthen the enforcement of sanctions with regard to road-safety-related traffic offences and to propose common criteria concerning the follow-up procedures in the case of non-payment of a financial penalty, within the framework of all relevant Union policies, including the common transport policy,  the possibilities for harmonising traffic rules where appropriate,  an assessment of the software applications as referred to in Article 4(4), with a view to ensuring proper implementation of this Directive as well as guaranteeing an effective, expeditious, secure and confidential exchange of specific VRD. Article 12 Transposition 1. Member States shall bring into force the laws, regulations and administrative provisions necessary to comply with this Directive by 6 May 2015. They shall forthwith communicate to the Commission the text of those provisions. When Member States adopt those provisions, they shall contain a reference to this Directive or be accompanied by such a reference on the occasion of their official publication. Member States shall determine how such reference is to be made. By way of derogation from the first subparagraph, the Kingdom of Denmark, Ireland and the United Kingdom of Great Britain and Northern Ireland may postpone the deadline referred to in the first subparagraph until 6 May 2017. 2. Member States shall communicate to the Commission the text of the main provisions of national law which they adopt in the field covered by this Directive. Article 13 Entry into force This Directive shall enter into force on the fourth day following that of its publication in the Official Journal of the European Union. Article 14 Addressees This Directive is addressed to the Member States. Done at Strasbourg, 11 March 2015. For the European Parliament The President M. SCHULZ For the Council The President Z. KALNIÃ A-LUKAÃ EVICA (1) OJ C 12, 15.1.2015, p. 115. (2) Position of the European Parliament of 11 February 2015 (not yet published in the Official Journal) and Decision of the Council of 2 March 2015. (3) Council Decision 2008/615/JHA of 23 June 2008 on the stepping up of cross-border cooperation, particularly in combating terrorism and cross-border crime (OJ L 210, 6.8.2008, p. 1). (4) Council Decision 2008/616/JHA of 23 June 2008 on the implementation of Decision 2008/615/JHA on the stepping up of cross-border cooperation, particularly in combating terrorism and cross-border crime (OJ L 210, 6.8.2008, p. 12). (5) Directive 2011/82/EU of the European Parliament and of the Council of 25 October 2011 facilitating the cross-border exchange of information on road safety related traffic offences (OJ L 288, 5.11.2011, p. 1). (6) Judgment in Commission v Parliament and Council, C-43/12, EU:C:2014:298. (7) Council Framework Decision 2005/214/JHA of 24 February 2005 on the application of the principle of mutual recognition to financial penalties (OJ L 76, 22.3.2005, p. 16). (8) Directive 2010/64/EU of the European Parliament and of the Council of 20 October 2010 on the right to interpretation and translation in criminal proceedings (OJ L 280, 26.10.2010, p. 1). (9) Directive 95/46/EC of the European Parliament and of the Council of 24 October 1995 on the protection of individuals with regard to the processing of personal data and on the free movement of such data (OJ L 281, 23.11.1995, p. 31). (10) Regulation (EC) No 45/2001 of the European Parliament and of the Council of 18 December 2000 on the protection of individuals with regard to the processing of personal data by the Community institutions and bodies and on the free movement of such data (OJ L 8, 12.1.2001, p. 1). (11) Council Directive 91/671/EEC of 16 December 1991 relating to the compulsory use of safety belts and child-restraint systems in vehicles (OJ L 373, 31.12.1991, p. 26). ANNEX I Data elements necessary to conduct the search referred to in Article 4(1) Item M/O (1) Remarks Data relating to the vehicle M Member State of registration M Registration number M (A (2)) Data relating to the offence M Member State of the offence M Reference date of the offence M Reference time of the offence M Purpose of the search M Code indicating the type of offence as listed in Article 2 1. = Speeding 2. = Drink-driving 3. = Failing to use a seat belt 4. = Failing to stop at a red traffic light 5. = Use of a forbidden lane 10. = Driving under the influence of drugs 11. = Failing to wear a safety helmet 12. = Illegally using a mobile phone or any other communication devices while driving Data elements provided as a result of the search conducted pursuant to Article 4(1) Part I. Data relating to vehicles Item M/O (3) Remarks Registration number M Chassis number/VIN M Member State of registration M Make M (D.1 (4)) e.g. Ford, Opel, Renault Commercial type of the vehicle M (D.3) e.g. Focus, Astra, Megane EU Category Code M (J) e.g. mopeds, motorbikes, cars Part II. Data relating to owners or holders of the vehicles Item M/O (5) Remarks Data relating to holders of the vehicle (C.1 (6)) The data refer to the holder of the specific registration certificate. Registration holders' (company) name M (C.1.1) Separate fields shall be used for surname, infixes, titles, etc., and the name in printable format shall be communicated. First name M (C.1.2) Separate fields for first name(s) and initials shall be used, and the name in printable format shall be communicated. Address M (C.1.3) Separate fields shall be used for street, house number and annex, post code, place of residence, country of residence, etc., and the address in printable format shall be communicated. Gender O Male, female Date of birth M Legal entity M Individual, association, company, firm, etc. Place of Birth O ID Number O An identifier that uniquely identifies the person or the company. Data relating to owners of the vehicle (C.2) The data refer to the owner of the vehicle. Owners' (company) name M (C.2.1) First name M (C.2.2) Address M (C.2.3) Gender O Male, female Date of birth M Legal entity M Individual, association, company, firm, etc. Place of Birth O ID Number O An identifier that uniquely identifies the person or the company. In case of scrap vehicles, stolen vehicles or number plates, or outdated vehicle registration no owner/holder information shall be provided. Instead, the message Information not disclosed shall be returned. (1) M = mandatory when available in national register, O = optional. (2) Harmonised code, see Council Directive 1999/37/EC of 29 April 1999 on the registration documents for vehicles (OJ L 138, 1.6.1999, p. 57). (3) M = mandatory when available in national register, O = optional. (4) Harmonised code, see Directive 1999/37/EC. (5) M = mandatory when available in national register, O = optional. (6) Harmonised code, see Directive 1999/37/EC. ANNEX II Text of image TEMPLATE FOR THE INFORMATION LETTER referred to in Article 5 [Cover page] [Name, address and telephone number of sender] [Name and address of addressee] INFORMATION LETTER regarding a road-safety-related traffic offence committed in [name of the Member State of the offence] Text of image Page 2 On a road-safety-related traffic offence committed with the vehicle with registration [date] number make model was detected by [name of the responsible body] [Option 1] (1) You are registered as the holder of the registration certificate of the abovementioned vehicle. [Option 2] (1) The holder of the registration certificate of the abovementioned vehicle indicated that you were driving that vehicle when the road-safety-related traffic offence was committed. The relevant details of the offence are described on page 3 below. The amount of the financial penalty due for this offence is EUR/national currency. Deadline for the payment is You are advised to complete the attached reply form (page 4) and send it to the address shown, if you do not pay this financial penalty. This letter shall be processed in accordance with the national law of [name of the Member State of the offence]. Text of image Page 3 Relevant details concerning the offence (a) Data concerning the vehicle with which the offence was committed: Registration number: Member State of registration: Make and model: (b) Data concerning the offence: Place, date and time where the offence was committed: Nature and legal classification of the offence: speeding, failing to use a seatbelt, failing to stop at a red traffic light, drink-driving, driving under the influence of drugs, failing to wear a safety helmet, use of a forbidden lane, illegally using a mobile telephone or any other communication devices while driving (1) Detailed description of the offence: Reference to the relevant legal provision(s): Description of or reference to the evidence for the offence: Text of image (c) Data concerning the device that was used for detecting the offence (2): Type of device for detection of speeding, failing to use a seatbelt, failing to stop at a red traffic light, drink-driving, driving under the influence of drugs, failing to wear a safety helmet, use of a forbidden lane, illegally using a mobile telephone or any other communication devices while driving (1): Specification of the device: Identification number of the device: Expiry date for the last gauging: (d) The result of the application of the device: [example for speeding; other offences to be added:] The maximum speed: The measured speed: The measured speed corrected for margin of error: (1) Delete if not applicable. (2) Not applicable if no device has been used. Text of image Page 4 Reply form (please complete using block capitals) A. Identity of the driver:  Full name:  Place and date of birth:  Number of driving licence: delivered (date): and at (place):  Address: B. List of questions: 1. Is the vehicle, make , registration number , registered in your name? yes/no (1) If not, the holder of the registration certificate is: (name, first name, address) 2. Do you acknowledge that you committed the offence? yes/no (1) 3. If you do not acknowledge this, please explain why: Please send the completed form within 60 days from the date of this information letter to the following authority: at the following address: INFORMATION This case will be examined by the competent authority of [name of the Member State of the offence] If this case is not pursued, you will be informed within 60 days after receipt of the reply form. (1) Delete if not applicable. Text of image If this case is pursued, the following procedure applies: [to be filled in by the Member State of the offence  what the further procedure will be, including details of the possibility and procedure of appeal against the decision to pursue the case. These details shall in any event include: name and address of the authority in charge of pursuing the case; deadline for payment; name and address of the body of appeal concerned; deadline for appeal]. This letter as such does not lead to legal consequences.
============================== "END OF DOC" ==============================
        

        doc 3 :

        'celex': 32006L0126
        'status': In Force
        'act_type': Directive
        'treaty': TEC (1992)

        full_doc :

        30.12.2006 EN Official Journal of the European Union L 403/18 DIRECTIVE 2006/126/EC OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 20 December 2006 on driving licences (Recast) (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty establishing the European Community, and in particular Article 71 thereof, Having regard to the proposal from the Commission, Having regard to the opinion of the European Economic and Social Committee (1), After consulting the Committee of the Regions, Acting in accordance with the procedure laid down in Article 251 of the Treaty (2), Whereas: (1) Council Directive 91/439/EEC of 29 July 1991 on driving licences (3) has been significantly amended on several occasions. Now that new amendments are being made to the said Directive, it is desirable, in order to clarify matters, that the provisions in question should be recast. (2) The rules on driving licences are essential elements of the common transport policy, contribute to improving road safety, and facilitate the free movement of persons taking up residence in a Member State other than the one issuing the licence. Given the importance of individual means of transport, possession of a driving licence duly recognised by a host Member State promotes free movement and freedom of establishment of persons. Despite the progress achieved with harmonising the rules on driving licences, significant differences have persisted between Member States in the rules on periodicity of licences renewal and on subcategories of vehicles, which needed to be harmonised more fully, in order to contribute to the implementation of Community policies. (3) The possibility of laying down national provisions with regard to the period of validity provided for in Directive 91/439/EEC leads to the co-existence of different rules in different Member States and over 110 different models of driving licences valid in the Member States. This creates problems of transparency for citizens, police forces and the administrations responsible for the administration of driving licences and leads to the falsification of documents which sometimes date back several decades. (4) In order to prevent the single European driving licence model from becoming an additional model to the 110 already in circulation, Member States should take all necessary measures to issue this single model to all licence holders. (5) This Directive should not prejudice existing entitlements to drive granted or acquired before its date of application. (6) Driving licences are mutually recognised. Member States should be able to apply the period of validity prescribed by this Directive to a licence without a limited administrative validity issued by another Member State and whose holder has resided on their territory for more than two years. (7) The introduction of a period of administrative validity for new driving licences should make it possible to apply at the time of periodic renewal the most recent counter-falsification measures and the medical examinations or other measures provided for by the Member States. (8) On road safety grounds, the minimum requirements for the issue of a driving licence should be laid down. Standards for driving tests and licensing need to be harmonised. To this end the knowledge, skills and behaviour connected with driving motor vehicles should be defined, the driving test should be based on these concepts and the minimum standards of physical and mental fitness for driving such vehicles should be redefined. (9) Proof of fulfilment of compliance with minimum standards of physical and mental fitness for driving by drivers of vehicles used for the transport of persons or goods should be provided when the driving licence is issued and periodically thereafter. Such regular control in accordance with national rules of compliance with minimum standards will contribute to the free movement of persons, avoid distortions of competition and better take into account the specific responsibility of drivers of such vehicles. Member States should be allowed to impose medical examinations as a guarantee of compliance with the minimum standards of physical and mental fitness for driving other motor vehicles. For reasons of transparency, such examinations should coincide with a renewal of driving licences and therefore be determined by the period of validity of the licence. (10) It is necessary to strengthen further the principle of progressive access to the categories of two-wheeled vehicles and to the categories of vehicles used for the transport of passengers and goods. (11) Nevertheless, Member States should be allowed to set a higher age limit for the driving of certain categories of vehicles in order to further promote road safety; Member States should in exceptional circumstances be allowed to set lower age limits in order to take account of national circumstances. (12) The definitions of the categories should reflect to a greater extent the technical characteristics of the vehicles concerned and the skills needed to drive a vehicle. (13) Introducing a category of driving licences for mopeds will, in particular, increase road safety as regards the youngest drivers who, according to the statistics, are the hardest hit by road accidents. (14) Specific provisions should be adopted to make it easier for physically disabled persons to drive vehicles. (15) For reasons connected with road safety, Member States should be able to apply their national provisions on the withdrawal, suspension, renewal and cancellation of driving licences to all licence holders having acquired normal residence in their territory. (16) The model driving licence as set out in Directive 91/439/EEC should be replaced by a single model in the form of a plastic card. At the same time, this model driving licence needs to be adapted on account of the introduction of a new category of driving licences for mopeds and of a new category of driving licences for motorcycles. (17) The introduction of an optional microchip in the new plastic card model driving licence should enable the Member States to further improve the level of anti-fraud protection. Member States should have flexibility to include national data on the chip provided that it does not interfere with commonly accessible data. The technical requirements for the microchip should be determined by the Commission, assisted by the committee on driving licences. (18) Minimum standards concerning access to the profession of examiner and examiner training requirements should be established in order to improve the knowledge and skills of examiners thereby ensuring a more objective evaluation of driving licence applicants and achieving greater harmonisation of driving tests. (19) The Commission should be allowed to undertake the adaptation of Annexes I to VI to scientific and technical progress. (20) The measures necessary for the implementation of this Directive should be adopted in accordance with Council Decision 1999/468/EC of 28 June 1999 laying down the procedures for the exercise of implementing powers conferred on the Commission (4). (21) In particular, the Commission should be empowered to establish the criteria necessary for the application of this Directive. Since those measures are of general scope and are designed to amend non-essential elements of this Directive, they should be adopted in accordance with the regulatory procedure with scrutiny provided for in Article 5a of Decision 1999/468/EC. (22) Since the objectives of this Directive cannot be sufficiently achieved by the Member States and can therefore, by reason of their scale and their effects, be better achieved at Community level, the Community may adopt measures, in accordance with the principle of subsidiarity as set out in Article 5 of the Treaty. In accordance with the principle of proportionality, as set out in that Article, this Directive does not go beyond what is necessary in order to achieve those objectives. (23) This Directive should not prejudice the obligations of the Member States relating to the deadlines for transposition into national law and application of the Directives listed in Annex VII, Part B, HAVE ADOPTED THIS DIRECTIVE: Article 1 Model licence 1. Member States shall introduce a national driving licence based on the Community model set out in Annex I, in accordance with the provisions of this Directive. The emblem on page 1 of the Community model driving licences shall contain the distinguishing sign of the Member State issuing the licence. 2. Without prejudice to data protection rules, Member States may introduce a storage medium (microchip) as part of the driving licence, as soon as the requirements concerning the microchip referred to in Annex I, which are designed to amend non-essential elements of this Directive, by supplementing it, are laid down by the Commission in accordance with the procedure referred to in Article 9(2). These requirements shall provide for EC type-approval, which shall only be granted when the ability to resist attempts to tamper with or alter data is demonstrated. 3. The microchip shall incorporate the harmonised driving licence data specified in Annex I. After consulting the Commission, Member States may store additional data, provided that it does not in any way interfere with the implementation of this Directive. In accordance with the procedure referred to in Article 9(2), the Commission may amend Annex I in order to guarantee future interoperability. 4. With the agreement of the Commission, Member States may make to the model set out in Annex I such adjustments as are necessary for computer processing of the driving licence. Article 2 Mutual recognition 1. Driving licences issued by Member States shall be mutually recognised. 2. When the holder of a valid national driving licence without the administrative validity period set out in Article 7(2) takes up normal residence in a Member State other than that which issued the driving licence, the host Member State may apply to the licence the administrative validity periods set out in that Article by renewing the driving licence, as from 2 years after the date on which the holder has taken up normal residence on its territory. Article 3 Anti-forgery measures 1. Member States shall take all necessary steps to avoid any risk of forgery of driving licences, including that of model driving licences issued before the entry into force of this Directive. They shall inform the Commission thereof. 2. The material used for the driving licence, as set out in Annex I, shall be made secure against forgery in application of specifications designed to amend non-essential elements of this Directive, by supplementing it, which are to be laid down by the Commission in accordance with the procedure referred to in Article 9(2). Member States are free to introduce additional security features. 3. Member States shall ensure that, by 19 January 2033, all driving licences issued or in circulation fulfil all the requirements of this Directive. Article 4 Categories, definitions and minimum ages 1. The driving licence provided for in Article 1 shall authorise the driving of power-driven vehicles in the categories defined hereafter. It may be issued from the minimum age indicated for each category. A power-driven vehicle means any self-propelled vehicle running on a road under its own power, other than a rail-borne vehicle. 2. mopeds: Category AM:  Two-wheel vehicles or three-wheel vehicles with a maximum design speed of not more than 45 km/h, as defined in Article 1(2)(a) of Directive 2002/24/EC of the European Parliament and of the Council of 18 March 2002 relating to the type-approval of two or three-wheel motor vehicles (5) (excluding those with a maximum design speed under or equal to 25 km/h), and light quadricycles as defined in Article 1(3)(a) of Directive 2002/24/EC,  the minimum age for category AM is fixed at 16 years; 3. motorcycles with or without a sidecar and motor tricycles:  motorcycle means two-wheel vehicles with or without a sidecar, as defined in Article 1(2)(b) of Directive 2002/24/EC,  motor tricycle means vehicles with three symmetrically arranged wheels, as defined in Article 1(2)(c) of Directive 2002/24/EC; (a) Category A1:  motorcycles with a cylinder capacity not exceeding 125 cubic centimetres, of a power not exceeding 11 kW and with a power/weight ratio not exceeding 0,1 kW/kg,  motor tricycles with a power not exceeding 15 kW,  the minimum age for category A1 is fixed at 16 years; (b) Category A2:  motorcycles of a power not exceeding 35 kW and with a power/weight ratio not exceeding 0,2 kW/kg and not derived from a vehicle of more than double its power,  the minimum age for category A2 is fixed at 18 years; (c) Category A: (i) motorcycles  The minimum age for category A is fixed at 20 years. However, access to the driving of motorcycles of this category shall be subject to a minimum of two years' experience on motorcycles under an A2 licence. This requirement as to previous experience may be waived if the candidate is at least 24 years old. (ii) motor tricycles with a power exceeding 15 kW  The minimum age for motor tricycles exceeding 15 kW is fixed at 21 years. 4. motor vehicles:  motor vehicle means any power-driven vehicle, which is normally used for carrying persons or goods by road or for drawing, on the road, vehicles used for the carriage of persons or goods. This term shall include trolleybuses, i.e. vehicles connected to an electric conductor and not rail-borne. It shall not include agricultural or forestry tractors,  Agricultural or forestry tractor means any power-driven vehicle running on wheels or tracks, having at least two axles, the principal function of which lies in its tractive power, which is specially designed to pull, push, carry or operate certain tools, machines or trailers used in connection with agricultural or forestry operations, and the use of which for carrying persons or goods by road or drawing, on the road, vehicles used for the carriage of persons or goods is only a secondary function; (a) Category B1:  quadricycles, as defined in Article 1(3)(b) of Directive 2002/24/EC,  the minimum age for category B1 is fixed at 16 years,  category B1 is optional; in Member States which do not introduce this category of driving licence, a driving licence for category B shall be required to drive such vehicles; (b) Category B: motor vehicles with a maximum authorised mass not exceeding 3 500 kg and designed and constructed for the carriage of no more than eight passengers in addition to the driver; motor vehicles in this category may be combined with a trailer having a maximum authorised mass which does not exceed 750 kg. Without prejudice to the provisions of type-approval rules for the vehicles concerned, motor vehicles in this category may be combined with a trailer with a maximum authorised mass exceeding 750 kg, provided that the maximum authorised mass of this combination does not exceed 4 250 kg. In case such a combination exceeds 3 500 kg, Member States shall, in accordance with the provisions of Annex V, require that this combination shall only be driven after:  a training has been completed, or  a test of skills and behaviour has been passed. Member States may also require both such a training and the passing of a test of skills and behaviour. Member States shall indicate the entitlement to drive such a combination on the driving licence by means of the relevant Community code. The minimum age for category B is fixed at 18 years; (c) Category BE:  without prejudice to the provisions of type-approval rules for the vehicles concerned, combination of vehicles consisting of a tractor vehicle in category B and a trailer or semi-trailer where the maximum authorised mass of the trailer or semi-trailer does not exceed 3 500 kg,  the minimum age for category BE is fixed at 18 years; (d) Category C1: motor vehicles other than those in categories D1 or D, the maximum authorised mass of which exceeds 3 500 kg, but does not exceed 7 500 kg, and which are designed and constructed for the carriage of no more than eight passengers in addition to the driver; motor vehicles in this category may be combined with a trailer having a maximum authorised mass not exceeding 750 kg; (e) Category C1E:  without prejudice to the provisions of type-approval rules for the vehicles concerned, combinations of vehicles where the tractor vehicle is in category C1 and its trailer or semi-trailer has a maximum authorised mass of over 750 kg provided that the authorised mass of the combination does not exceed 12 000 kg,  without prejudice to the provisions of type-approval rules for the vehicles concerned, combinations of vehicles where the tractor vehicle is in category B and its trailer or semi-trailer has an authorised mass of over 3 500 kg, provided that the authorised mass of the combination does not exceed 12 000 kg,  the minimum age for categories C1 and C1E is fixed at the age of 18 years, without prejudice to the provisions for the driving of such vehicles in Directive 2003/59/EC of the European Parliament and of the Council of 15 July 2003 on the initial qualification and periodic training of drivers of certain road vehicles for the carriage of goods or passengers (6); (f) Category C: motor vehicles other than those in categories D1 or D, whose maximum authorised mass is over 3 500 kg and which are designed and constructed for the carriage of no more than eight passengers in addition to the driver; motor vehicles in this category may be combined with a trailer having a maximum authorised mass which does not exceed 750 kg; (g) Category CE:  without prejudice to the provisions of type-approval rules for the vehicles concerned, combinations of vehicles where the tractor vehicle is in category C and its trailer or semi-trailer has a maximum authorised mass of over 750 kg,  the minimum age for categories C and CE is fixed at 21 years, without prejudice to the provisions for the driving of such vehicles in Directive 2003/59/EC; (h) Category D1: motor vehicles designed and constructed for the carriage of no more than 16 passengers in addition to the driver and with a maximum length not exceeding 8 m; motor vehicles in this category may be combined with a trailer having a maximum authorised mass not exceeding 750 kg; (i) Category D1E:  without prejudice to the provisions of type-approval rules for the vehicles concerned, combinations of vehicles where the tractor vehicle is in category D1 and its trailer has a maximum authorised mass of over 750 kg,  the minimum age for categories D1 and D1E is fixed at 21 years, without prejudice to the provisions for the driving of such vehicles in Directive 2003/59/EC; (j) Category D: motor vehicles designed and constructed for the carriage of more than eight passengers in addition to the driver; motor vehicles which may be driven with a category D licence may be combined with a trailer having a maximum authorised mass which does not exceed 750 kg; (k) Category DE:  without prejudice to the provisions of type-approval rules for the vehicles concerned, combinations of vehicles where the tractor vehicle is in category D and its trailer has a maximum authorised mass of over 750 kg,  the minimum age for categories D and DE is fixed at 24 years, without prejudice to the provisions for the driving of such vehicles in Directive 2003/59/EC; 5. With the agreement of the Commission, Member States may exclude from the application of this Article certain specific types of power-driven vehicle such as special vehicles for disabled persons. Member States may exclude from the application of this Directive vehicles used by, or under the control of, the armed forces and civil defence. 6. Member States may raise or lower the minimum age for issuing a driving licence: (a) for category AM down to 14 years or up to 18 years; (b) for category B1 up to 18 years; (c) for category A1 up to 17 or 18 years,  if there is a two years difference between the minimum age for category A1 and the minimum age for category A2, and  there is a requirement of a minimum of two years experience on motorcycles of category A2 before access to the driving of motorcycles for category A can be granted, as referred to in Article 4(3)(c)(i); (d) for categories B and BE down to 17 years. Member States may lower the minimum age for category C to 18 years and for category D to 21 years with regard to: (a) vehicles used by the fire service and vehicles used for maintaining public order; (b) vehicles undergoing road tests for repair or maintenance purposes. Driving licences issued to persons at a lower age than set out in paragraphs 2 to 4 in accordance with this paragraph shall only be valid on the territory of the issuing Member State until the licence holder has reached the minimum age limit set out in paragraphs 2 to 4. Member States may recognise the validity on their territory of driving licences issued to drivers under the minimum ages set out in paragraphs 2 to 4. Article 5 Conditions and restrictions 1. Driving licences shall state the conditions under which the driver is authorised to drive. 2. If, because of a physical disability, driving is authorised only for certain types of vehicle or for adapted vehicles, the test of skills and behaviour provided for in Article 7 shall be taken in such a vehicle. Article 6 Staging and equivalences between categories 1. The issue of driving licences shall be subject to the following conditions: (a) licences for categories C1, C, D1 and D shall be issued only to drivers already entitled to drive vehicles in category B; (b) licences for categories BE, C1E, CE, D1E and DE shall be issued only to drivers already entitled to drive vehicles in categories B, C1, C, D1 and D respectively. 2. The validity of driving licences shall be determined as follows: (a) licences granted for categories C1E, CE, D1E or DE shall be valid for combinations of vehicles in category BE; (b) licences granted for category CE shall be valid for category DE as long as their holders are entitled to drive vehicles in category D; (c) licences granted for category CE and DE shall be valid for combinations of vehicles in categories C1E and D1E respectively; (d) licences granted for any category shall be valid for vehicles in category AM. However, for driving licences issued on its territory, a Member State may limit the equivalences for category AM to categories A1, A2 and A, if that Member State imposes a practical test as a condition for obtaining category AM; (e) licences issued for category A2 shall also be valid for category A1; (f) licences granted for categories A, B, C or D shall be valid for categories A1, A2, B1, C1, or D1 respectively. 3. For driving on their territory, Member States may grant the following equivalences: (a) motor tricycles under a licence for category B, for motor tricycles with a power exceeding 15 kW provided that the holder of the licence for category B is at least 21 years old; (b) category A1 motorcycles under a licence for category B. As this paragraph is only valid on their territories, Member States shall not indicate on the driving licence that a holder is entitled to drive these vehicles. 4. Member States may, after consulting the Commission, authorise the driving on their territory of: (a) vehicles of category D1 (with a maximum authorised mass of 3 500 kg, excluding any specialised equipment intended for the carriage of disabled passengers) by holders over 21 years old of a driving licence for category B which was obtained at least two years earlier provided that the vehicles are being used by non-commercial bodies for social purposes and that the driver provides his services on a voluntary basis; (b) vehicles of a maximum authorised mass exceeding 3 500 kg by holders over 21 years old of a driving licence for category B which was obtained at least two years before, provided that the main purpose of the vehicles is to be used only when stationary as an instructional or recreational area, and that they are being used by non-commercial bodies for social purposes and that vehicles have been modified so that they may not be used either for the transport of more than nine persons or for the transport of any goods other than those strictly necessary for their purposes. Article 7 Issue, validity and renewal 1. Driving licences shall be issued only to those applicants: (a) who have passed a test of skills and behaviour and a theoretical test and who meet medical standards, in accordance with the provisions of Annexes II and III; (b) who have passed a theory test only as regards category AM; Member States may require applicants to pass a test of skills and behaviour and a medical examination for this category. For tricycles and quadricycles within this category, Member States may impose a distinctive test of skills and behaviour. For the differentiation of vehicles in category AM, a national code may be inserted on the driving licence; (c) who have, as regards category A2 or category A, on the condition of having acquired a minimum of 2 years' experience on a motorcycle in category A1 or in category A2 respectively, passed a test of skills and behaviour only, or completed a training pursuant to Annex VI; (d) who have completed a training or passed a test of skills and behaviour, or completed a training and passed a test of skills and behaviour pursuant to Annex V as regards category B for driving a vehicle combination as defined in the second subparagraph of Article 4(4)(b); (e) who have their normal residence in the territory of the Member State issuing the licence, or can produce evidence that they have been studying there for at least six months. 2. (a) As from 19 January 2013, licences issued by Member States for categories AM, A1, A2, A, B, B1 and BE shall have an administrative validity of 10 years. A Member State may choose to issue such licences with an administrative validity of up to 15 years; (b) As from 19 January 2013, licences issued by Member States for categories C, CE, C1, C1E, D, DE, D1, D1E shall have an administrative validity of 5 years; (c) The renewal of a driving licence may trigger a new administrative validity period for another category or categories the licence holder is entitled to drive, insofar as this is in conformity with the conditions laid down in this Directive; (d) The presence of a microchip pursuant to Article 1 shall not be a prerequisite for the validity of a driving licence. The loss or unreadability of the microchip, or any other damage thereto, shall not affect the validity of the document. 3. The renewal of driving licences when their administrative validity expires shall be subject to: (a) continuing compliance with the minimum standards of physical and mental fitness for driving set out in Annex III for driving licences in categories C, CE, C1, C1E, D, DE, D1, D1E; and (b) normal residence in the territory of the Member State issuing the licence, or evidence that applicants have been studying there for at least six months. Member States may, when renewing driving licences in categories AM, A, A1, A2, B, B1 and BE, require an examination applying the minimum standards of physical and mental fitness for driving set out in Annex III. Member States may limit the period of administrative validity set out in paragraph 2 of driving licences issued to novice drivers for any category in order to apply specific measures to such drivers, aiming at improving road safety. Member States may limit the period of administrative validity of the first licence issued to novice drivers for categories C and D to 3 years in order to be able to apply specific measures to such drivers, so as to improve their road safety. Member States may limit the period of administrative validity set out in paragraph 2 of individual driving licences for any category in case it is found necessary to apply an increased frequency of medical checks or other specific measures such as restrictions for traffic offenders. Member States may reduce the period of administrative validity set out in paragraph 2 of driving licences of holders residing on their territory having reached the age of 50 years in order to apply an increased frequency of medical checks or other specific measures such as refresher courses. This reduced period of administrative validity can only be applied upon renewing the driving licence. 4. Without prejudice to national criminal and police laws, Member States may, after consulting the Commission, apply to the issuing of driving licences the provisions of their national rules relating to conditions other than those referred to in this Directive. 5. (a) No person may hold more than one driving licence; (b) A Member State shall refuse to issue a licence where it establishes that the applicant already holds a driving licence; (c) Member States shall take the necessary measures pursuant to point (b). The necessary measures as regards the issue, replacement, renewal or exchange of a driving licence shall be to verify with other Member States where there are reasonable grounds to suspect that the applicant is already the holder of another driving licence; (d) In order to facilitate the checks pursuant to point (b), Member States shall use the EU driving licence network once it is operational. Without prejudice to Article 2, a Member State issuing a licence shall apply due diligence to ensure that a person fulfils the requirements set out in paragraph 1 of this Article and shall apply its national provisions on the cancellation or withdrawal of the right to drive if it is established that a licence has been issued without the requirements having been met. Article 8 Adaptation to scientific and technical progress The amendments necessary to adapt Annexes I to VI to scientific and technical progress shall be adopted in accordance with the procedure referred to in Article 9(2). Article 9 Committee 1. The Commission shall be assisted by the committee on driving licences. 2. Where reference is made to this paragraph, Article 5a(1) to (4), and Article 7 of Decision 1999/468/EC shall apply, having regard to the provisions of Article 8 thereof. Article 10 Examiners From the entry into force of this Directive, driving examiners shall meet the minimum standards set out in Annex IV. Driving examiners already working in that capacity before 19 January 2013 shall be subject only to the requirements concerning quality assurance and regular periodic training measures. Article 11 Various provisions concerning the exchange, the withdrawal, the replacement and the recognition of driving licences 1. Where the holder of a valid national driving licence issued by a Member State has taken up normal residence in another Member State, he may request that his driving licence be exchanged for an equivalent licence. It shall be for the Member State effecting the exchange to check for which category the licence submitted is in fact still valid. 2. Subject to observance of the principle of territoriality of criminal and police laws, the Member State of normal residence may apply its national provisions on the restriction, suspension, withdrawal or cancellation of the right to drive to the holder of a driving licence issued by another Member State and, if necessary, exchange the licence for that purpose. 3. The Member State effecting the exchange shall return the old licence to the authorities of the Member State which issued it and give the reasons for doing so. 4. A Member State shall refuse to issue a driving licence to an applicant whose driving licence is restricted, suspended or withdrawn in another Member State. A Member State shall refuse to recognise the validity of any driving licence issued by another Member State to a person whose driving licence is restricted, suspended or withdrawn in the former State's territory. A Member State may also refuse to issue a driving licence to an applicant whose licence is cancelled in another Member State. 5. A replacement for a driving licence which has, for example, been lost or stolen may only be obtained from the competent authorities of the Member State in which the holder has his normal residence; those authorities shall provide the replacement on the basis of the information in their possession or, where appropriate, proof from the competent authorities of the Member State which issued the original licence. 6. Where a Member State exchanges a driving licence issued by a third country for a Community model driving licence, such exchange shall be recorded on the Community model driving licence as shall any subsequent renewal or replacement. Such an exchange may occur only if the licence issued by the third country has been surrendered to the competent authorities of the Member State making the exchange. If the holder of this licence transfers his normal residence to another Member State, the latter need not apply the principle of mutual recognition set out in Article 2. Article 12 Normal residence For the purpose of this Directive, normal residence means the place where a person usually lives, that is for at least 185 days in each calendar year, because of personal and occupational ties, or, in the case of a person with no occupational ties, because of personal ties which show close links between that person and the place where he is living. However, the normal residence of a person whose occupational ties are in a different place from his personal ties and who consequently lives in turn in different places situated in two or more Member States shall be regarded as being the place of his personal ties, provided that such person returns there regularly. This last condition need not be met where the person is living in a Member State in order to carry out a task of a definite duration. Attendance at a university or school shall not imply transfer of normal residence. Article 13 Equivalences between non-Community model licences 1. With the agreement of the Commission, Member States shall establish equivalences between entitlements obtained before the implementation of this Directive and the categories defined in Article 4. After consulting the Commission, Member States may make to their national legislation such adjustments as are necessary for the purpose of implementing the provisions of Article 11(4), (5) and (6). 2. Any entitlement to drive granted before 19 January 2013 shall not be removed or in any way qualified by the provisions of this Directive. Article 14 Review The Commission shall report on the implementation of this Directive, including its impact on road safety, not earlier than 19 January 2018. Article 15 Mutual Assistance Member States shall assist one another in the implementation of this Directive and shall exchange information on the licences they have issued, exchanged, replaced, renewed or revoked. They shall use the EU driving licence network set up for these purposes, once this network is operational. Article 16 Transposition 1. Member States shall adopt and publish, not later than 19 January 2011, the laws, regulations and administrative provisions necessary to comply with Article 1(1), Article 3, Article 4(1), (2), (3) and (4)(b) to (k), Article 6(1), (2)(a), (c), (d) and (e), Article 7(1)(b), (c) and (d), (2), (3) and (5), Article 8, Article 10, Article 13, Article 14, Article 15, and Annexes I, point 2, II, point 5.2 concerning categories A1, A2 and A, IV, V and VI. They shall forthwith communicate to the Commission the text of those provisions. 2. They shall apply those provisions as from 19 January 2013. 3. When Member States adopt those provisions, they shall contain a reference to this Directive or shall be accompanied by such reference on the occasion of their official publication. They shall also contain an indication that references made, in the laws, regulations or administrative provisions in force, to the repealed Directive shall be construed as being made to this Directive. The methods of making such reference, and its wording, shall be laid down by Member States. 4. Member States shall communicate to the Commission the text of the main provisions of national law which they adopt in the field covered by this Directive. Article 17 Repeal Directive 91/439/EEC shall be repealed with effect from 19 January 2013, without prejudice to the obligations of the Member States with regard to the deadlines indicated in Annex VII, Part B for transposing that Directive into national law. Article 2(4) of Directive 91/439/EEC shall be repealed on 19 January 2007. References made to the repealed Directive shall be construed as being made to this Directive and should be read in accordance with the correlation table in Annex VIII. Article 18 Entry into force This Directive shall enter into force on the twentieth day following that of its publication in the Official Journal of the European Union. Article 2(1), Article 5, Article 6(2)(b), Article 7(1)(a), Article 9, Article 11(1), (3), (4), (5) and (6), Article 12, and Annexes I, II and III shall apply from 19 January 2009. Article 19 Addressees This Directive is addressed to the Member States. Done at Brussels, 20 December 2006. For the European Parliament The President J. BORRELL FONTELLES For the Council The President J. KORKEAOJA (1) OJ C 112, 30.4.2004, p. 34. (2) Opinion of the European Parliament of 23 February 2005 (OJ C 304 E, 1.12.2005, p. 202), Council Common Position of 18 September 2006 (OJ C 295 E, 5.12.2006, p. 1) and Position of the European Parliament of 14 December 2006 (not yet published in the Official Journal). Council Decision of 19 December 2006. (3) OJ L 237, 24.8.1991, p. 1. Directive as last amended by Regulation (EC) No 1882/2003 of the European Parliament and of the Council (OJ L 284, 31.10.2003, p. 1). (4) OJ L 184, 17.7.1999, p. 23. Decision as amended by Decision 2006/512/EC (OJ L 200, 22.7.2006, p. 11). (5) OJ L 124, 9.5.2002, p. 1. Directive as last amended by Commission Directive 2005/30/EC (OJ L 106, 27.4.2005, p. 17). (6) OJ L 226, 10.9.2003, p. 4. Directive as amended by Council Directive 2004/66/EC (OJ L 168, 1.5.2004, p. 35). ANNEX I PROVISIONS CONCERNING THE COMMUNITY MODEL DRIVING LICENCE 1. The physical characteristics of the card of the Community model driving licence shall be in accordance with ISO 7810 and ISO 7816-1. The card shall be made of polycarbonate. Methods for testing the characteristics of driving licences for the purpose of confirming their compliance with the international standards shall be in accordance with ISO 10373. 2. Physical security of driving licences The threats to the physical security of driving licences are:  production of false cards: creating a new object which bears great resemblance to the document, either by making it from scratch or by copying an original document,  material alteration: changing a property of an original document, e.g. modifying some of the data printed on the document; The overall security lies in the system in its entirety, consisting of the application process, the transmission of data, the card body material, the printing technique, a minimum set of different security features and the personalisation process. (a) The material used for driving licences shall be made secure against forgery by using the following techniques (mandatory security features):  card bodies shall be UV dull,  a security background pattern designed to be resistant to counterfeit by scanning, printing or copying, using rainbow printing with multicolour security inks and positive and negative guilloche printing. The pattern shall not be composed of the primary colours (CMYK), shall contain complex pattern designs in a minimum of two special colours and shall include micro lettering,  optical variable elements providing adequate protection against copying and tampering of the photograph,  laser engraving,  in the area of the photograph the security design background and photograph should overlap on at least its border (weakening pattern). (b) In addition, the material used for driving licences shall be made secure against forgery by using at least three of the following techniques (additional security features):  colour-shifting inks*,  termochromic ink*,  custom holograms*,  variable laser images*,  ultraviolet fluorescent ink, visible and transparent,  iridescent printing,  digital watermark in the background,  infrared or phosphorescent pigments,  tactile characters, symbols or patterns*. (c) Member States are free to introduce additional security features. As a basis, the techniques indicated with an asterisk are to be preferred as they enable the law enforcement officers to check the validity of the card without any special means. 3. The licence shall have two sides. Page 1 shall contain: (a) the words Driving Licence printed in large type in the language or languages of the Member State issuing the licence; (b) the name of the Member State issuing the licence (optional); (c) the distinguishing sign of the Member State issuing the licence, printed in negative in a blue rectangle and encircled by twelve yellow stars; the distinguishing signs shall be as follows: B : Belgium CZ : Czech Republic DK : Denmark D : Germany EST : Estonia GR : Greece E : Spain F : France IRL : Ireland I : Italy CY : Cyprus LV : Latvia LT : Lithuania L : Luxembourg H : Hungary M : Malta NL : The Netherlands A : Austria PL : Poland P : Portugal SLO : Slovenia SK : Slovakia FIN : Finland S : Sweden UK : The United Kingdom; (d) information specific to the licence issued, numbered as follows: 1. surname of the holder; 2. other name(s) of the holder; 3. date and place of birth; 4. (a) date of issue of the licence; (b) date of expiry of the licence or a dash if the licence is valid indefinitely under the provision of Article 7(2)(c); (c) the name of the issuing authority (may be printed on page 2); (d) a different number from the one under heading 5, for administrative purposes (optional); 5. number of the licence; 6. photograph of the holder; 7. signature of the holder; 8. permanent place of residence, or postal address (optional); 9. category of vehicle(s) the holder is entitled to drive (national categories shall be printed in a different type from harmonised categories); (e) the words European Communities model in the language(s) of the Member State issuing the licence and the words Driving Licence in the other languages of the Community, printed in pink to form the background of the licence: Permiso de ConducciÃ ³n Ã idiÃ skÃ ½ prÃ ¯kaz KÃ ¸rekort FÃ ¼hrerschein Juhiluba Ã Ã ´Ã µÃ ¹Ã ± Ã Ã ´Ã ®Ã ³Ã ·Ã Ã ·Ã  Driving Licence Permis de conduire CeadÃ ºas TiomÃ ¡na Patente di guida VadÃ «tÃ ja apliecÃ «ba Vairuotojo paÃ ¾ymÃ jimas VezetÃ i engedÃ ©ly LiÃ enzja tas-Sewqan Rijbewijs Prawo Jazdy Carta de ConduÃ §Ã £o VodiÃ skÃ ½ preukaz VozniÃ ¡ko dovoljenje Ajokortti KÃ ¶rkort; (f) Colour references:  blue: Pantone Reflex Blue,  yellow: Pantone Yellow. Page 2 shall contain: (a) 9. category of vehicle(s) the holder is entitled to drive (national categories shall be printed in a different type from harmonised categories); 10. date of first issue of each category (this date must be repeated on the new licence in the event of subsequent replacement or exchange); 11. date of expiry of each category; 12. additional information/restriction(s), in code form, facing the (sub)category affected. The codes shall be as follows:  codes 01 to 99 : harmonised Community codes DRIVER (Medical reasons) 01. Sight correction and/or protection 01.01 Glasses 01.02 Contact lense(s) 01.03 Protective glass 01.04 Opaque lense 01.05 Eye cover 01.06 Glasses or contact lenses 02. Hearing aid/communication aid 02.01 Hearing aid for one ear 02.02 Hearing aid for two ears 03. Prosthesis/orthosis for the limbs 03.01 Upper limb prosthesis/orthosis 03.02 Lower limb prosthesis/orthosis 05. Limited use (subcode use obligatory, driving subject to restrictions for medical reasons) 05.01 Limited to day time journeys (for example: one hour after sunrise and one hour before sunset) 05.02 Limited to journeys within a radius of ¦ km from holder's place of residence or only inside city/region 05.03 Driving without passengers 05.04 Limited to journeys with a speed not greater than ¦ km/h 05.05 Driving authorised solely when accompanied by a holder of a driving licence 05.06 Without trailer 05.07 No driving on motorways 05.08 No alcohol VEHICLE ADAPTATIONS 10. Modified transmission 10.01 Manual transmission 10.02 Automatic transmission 10.03 Electronically operated transmission 10.04 Adjusted gear-shift lever 10.05 Without secondary gearbox 15. Modified clutch 15.01 Adjusted gear-shift lever 15.02 Manual clutch 15.03 Automatic clutch 15.04 Partitioning in front of/fold away/detached clutch pedal 20. Modified braking systems 20.01 Adjusted brake pedal 20.02 Enlarged brake pedal 20.03 Brake pedal suitable for use by left foot 20.04 Brake pedal by sole 20.05 Tilted brake pedal 20.06 Manual (adapted) service brake 20.07 Maximum use of reinforced service brake 20.08 Maximum use of emergency brake integrated in the service brake 20.09 Adjusted parking brake 20.10 Electrically operated parking brake 20.11 (Adjusted) foot operated parking brake 20.12 Partitioning in front of/fold away/detached brake pedal 20.13 Brake operated by knee 20.14 Electrically operated service brake 25. Modified accelerator systems 25.01 Adjusted accelerator pedal 25.02 Accelerator pedal by sole 25.03 Tilted accelerator pedal 25.04 Manual accelerator 25.05 Accelerator at knee 25.06 Servo accelerator (electronic, pneumatic, etc.) 25.07 Accelerator pedal on the left of brake pedal 25.08 Accelerator pedal on the left 25.09 Partitioning in front of/fold away/detached accelerator pedal 30. Modified combined braking and accelerator systems 30.01 Parallel pedals 30.02 Pedals at (or almost at) the same level 30.03 Accelerator and brake with sliding 30.04 Accelerator and brake with sliding and orthesis 30.05 Fold away/detached accelerator and brake pedals 30.06 Raised floor 30.07 Partitioning on the side of the brake pedal 30.08 Partitioning for prosthesis on the side of the brake pedal 30.09 Partitioning in front of the accelerator and brake pedals 30.10 Heel/leg support 30.11 Electrically operated accelerator and brake 35. Modified control layouts (Lights switches, windscreen wiper/washer, horn, direction indicators, etc.) 35.01 Control devices operable without negative influence on the steering and handling 35.02 Control devices operable without releasing the steering wheel and accessories (knob, fork, etc.) 35.03 Control devices operable without releasing the steering wheel and accessories (knob, fork, etc.) with the left hand 35.04 Control devices operable without releasing the steering wheel and accessories (knob, fork, etc.) with the right hand 35.05 Control devices operable without releasing the steering wheel and accessories (knob, fork, etc.) and the combined accelerator and braking mechanismss 40. Modified steering 40.01 Standard assisted steering 40.02 Reinforced assisted steering 40.03 Steering with backup system 40.04 Lengthened steering column 40.05 Adjusted steering wheel (Larger and/or thicker steering wheel section, reduced diameter steering wheel, etc.) 40.06 Tilted steering wheel 40.07 Vertical steering wheel 40.08 Horizontal steering wheel 40.09 Foot operated driving 40.10 Alternative adjusted steering (joy-stick, etc.) 40.11 Knob on the steering wheel 40.12 Hand orthesis on the steering wheel 40.13 With orthesis tenodese 42. Modified rearview mirror(s) 42.01 External (left or) right-side rear-view mirror 42.02 External rear-view mirror set on the wing 42.03 Additional inside rear-view mirror permitting view of traffic 42.04 Panoramic inside rear-view mirror 42.05 Blind spot rear-view mirror 42.06 Electrically operated outside rear-view mirror(s) 43. Modified driver seat 43.01 Driver seat at a good viewing height and in normal distance from the steering wheel and the pedal 43.02 Driver seat adjusted to body shape 43.03 Driver seat with lateral support for good sitting stability 43.04 Driver seat with armrest 43.05 Lengthening of sliding driver's seat 43.06 Seat-belt adjustment 43.07 Harness-type seat-belt 44. Modifications to motorcycles (subcode use obligatory) 44.01 Single operated brake 44.02 (Adjusted) hand operated brake (front wheel) 44.03 (Adjusted) foot operated brake (back wheel) 44.04 (Adjusted) accelerator handle 44.05 (Adjusted) manual transmission and manual clutch 44.06 (Adjusted) rear-view mirror(s) 44.07 (Adjusted) commands (direction indicators, braking light, ¦) 44.08 Seat height allowing the driver, in sitting position, to have two feet on the road at the same time 45. Motorcycle with side-car only 50. Restricted to a specific vehicle/chassis number (vehicle identification number, VIN) 51. Restricted to a specific vehicle/registration plate (vehicle registration number, VRN) ADMINISTRATIVE MATTERS 70. Exchange of licence No ¦ issued by ¦ (EU/UN distinguishing sign in the case of a third country; e.g: 70.0123456789.NL) 71. Duplicate of licence No ¦ (EU/UN distinguishing sign in the case of a third country; e.g: 71.987654321.HR) 72. Restricted to category A vehicles having a maximum cylinder capacity of 125 cc and maximum power of 11 KW (A1) 73. Restricted to category B vehicles of the motor tricycle or quadricycle type (B1) 74. Restricted to category C vehicles the maximum authorised mass of which does not exceed 7 500 kg (C1) 75. Restricted to category D vehicles with not more than 16 seats, excluding the driver's seat (D1) 76. Restricted to category C vehicles the maximum authorised mass of which does not exceed 7 500 kg (C1), attached to a trailer the maximum authorised mass of which exceeds 750 kg, provided that the maximum authorised mass of the vehicle train thus formed does not exceed 12 000 kg, and that the maximum authorised mass of the trailer does not exceed the unladen mass of the drawing vehicle (C1E) 77. Restricted to category D vehicles with not more than 16 passenger seats, excluding the driver's seat (D1), attached to a trailer the maximum authorised mass of which exceeds 750 kg provided that (a) the maximum authorised mass of the vehicle train thus formed does not exceed 12 000 kg and the maximum authorised mass of the trailer does not exceed the unladen mass of the drawing vehicle and (b) the trailer is not used to carry passengers (D1E) 78. Restricted to vehicles with automatic transmission 79. ( ¦) Restricted to vehicles which comply with the specifications indicated in brackets, in the context of the application of Article 10(1) of Directive 91/439/EEC 90.01 : to the left 90.02 : to the right 90.03 : left 90.04 : right 90.05 : hand 90.06 : foot 90.07 : usable 95. Driver holding CPC meeting the obligation of professional aptitude provided for by Directive 2003/59/EC until ¦ [e.g.: 95.01.01.2012] 96. Driver having completed training or having passed a test of skills and behaviour in accordance with the provisions of Annex V.  codes 100 and above: : national codes valid only for driving in the territory of the Member State which issued the licence. Where a code applies to all categories for which the licence is issued, it may be printed under headings 9, 10 and 11; 13. in implementation of section 4(a) of this Annex, a space reserved for the possible entry by the host Member State of information essential for administering the licence; 14. a space reserved for the possible entry by the Member State which issues the licence of information essential for administering the licence or related to road safety (optional). If the information relates to one of the headings defined in this Annex, it should be preceded by the number of the heading in question. With the specific written agreement of the holder, information which is not related to the administration of the driving licence or road safety may also be added in this space; such addition shall not alter in any way the use of the model as a driving licence; (b) an explanation of the numbered items which appear on pages 1 and 2 of the licence (at least items 1, 2, 3, 4 (a), 4 (b), 4 (c), 5, 10, 11 and 12) If a Member State wishes to make the entries in a national language other than one of the following languages: Czech, Danish, Dutch, English, Estonian, Finnish, French, German, Greek, Hungarian, Italian, Latvian, Lithuanian, Maltese, Polish, Portuguese, Slovak, Slovenian, Spanish or Swedish, it shall draw up a bilingual version of the licence using one of the aforementioned languages, without prejudice to the other provisions of this Annex; (c) a space shall be reserved on the Community model licence to allow for the possible introduction of a microchip or similar computer device. 4. Special provisions (a) Where the holder of a driving licence issued by a Member State in accordance with this Annex has his normal place of residence in another Member State, that Member State may enter in the licence such information as is essential for administering it, provided that it also enters this type of information in the licences which it issues and provided that there remains enough space for the purpose. (b) After consulting the Commission, Member States may add colours or markings, such as bar codes and national symbols, without prejudice to the other provisions of this Annex. In the context of mutual recognition of licences, the bar code may not contain information other than what can already be read on the driving licence or which is essential to the process of issuing the licence. COMMUNITY MODEL DRIVING LICENCE Page 1 DRIVING LICENCE [MEMBER STATE] Page 2 1. Name 2. First name 3. Date and place of birth 4a. Date of issue of driving licence 4b. Official date of expiry 4c. Issued by 5. Serial number of licence 8. Place of residence 9. Category (1) 10. Date of issue, by category 11. Date of expiry, by category 12. Restrictions SPECIMEN MODEL LICENCE BELGIAN LICENCE (for information) (1) Note: a pictogram and a line for category AM will be added. Note: the term A2 will be added to the section on motorcycle categories. ANNEX II I. MINIMUM REQUIREMENTS FOR DRIVING TESTS Member States shall take the necessary measures to ensure that applicants for driving licences possess the knowledge and skills and exhibit the behaviour required for driving a motor vehicle. The tests introduced to this effect must consist of:  a theory test, and then  a test of skills and behaviour. The conditions under which these tests shall be conducted are set out below. A. THEORY TEST 1. Form The form chosen shall be such as to make sure that the applicant has the required knowledge of the subjects listed on points 2, 3 and 4. Any applicant for a licence in one category who has passed a theory test for a licence in a different category may be exempt from the common provisions of points 2, 3 and 4. 2. Content of the theory test concerning all vehicle categories 2.1. Questions must be asked on each of the points listed below, the content and form of the questions being left to the discretion of each Member State: 2.1.1. Road traffic regulations:  in particular as regards road signs, markings and signals, rights of way and speed limits; 2.1.2. The driver:  importance of alertness and of attitude to other road users,  perception, judgement and decision-taking, especially reaction time, as well as changes in driving behaviour due to the influence of alcohol, drugs and medicinal products, state of mind and fatigue; 2.1.3. The road:  the most important principles concerning the observance of a safe distance between vehicles, braking distances and road holding under various weather and road conditions,  driving risk factors related to various road conditions, in particular as they change with the weather and the time of day or night,  characteristics of various types of road and the related statutory requirements; 2.1.4. Other road users:  specific risk factors related to the lack of experience of other road users and the most vulnerable categories of users such as children, pedestrians, cyclists and people whose mobility is reduced,  risks involved in the movement and driving of various types of vehicles and of the different fields of view of their drivers; 2.1.5. General rules and regulations and other matters:  rules concerning the administrative documents required for the use of vehicles,  general rules specifying how the driver must behave in the event of an accident (setting warning devices and raising the alarm) and the measures which he can take to assist road accident victims where necessary,  safety factors relating to the vehicle, the load and persons carried; 2.1.6. Precautions necessary when alighting from the vehicle; 2.1.7. Mechanical aspects with a bearing on road safety; applicants must be able to detect the most common faults, in particular in the steering, suspension and braking systems, tyres, lights and direction indicators, reflectors, rear-view mirrors, windscreen and wipers, the exhaust system, seat-belts and the audible warning device; 2.1.8. Vehicle safety equipment and, in particular, the use of seat-belts, head restraints and child safety equipment; 2.1.9. Rules regarding vehicle use in relation to the environment (appropriate use of audible warning devices, moderate fuel consumption, limitation of pollutant emissions, etc.). 3. Specific provisions concerning categories A1, A2 and A 3.1. Compulsory check of general knowledge on: 3.1.1. Use of protective outfit such as gloves, boots, clothes and safety helmet; 3.1.2. Visibility of motorcycle riders for other road users; 3.1.3. Risk factors related to various road conditions as laid down above with additional attention to slippery parts such as drain covers, road markings such as lines and arrows, tram rails; 3.1.4. Mechanical aspects with a bearing on road safety as laid down above with additional attention to the emergency stop switch, the oil levels and the chain. 4. Specific provisions concerning categories C, CE, C1, C1E, D, DE, D1 and D1E 4.1. Compulsory check of general knowledge on: 4.1.1. Rules on driving hours and rest periods as defined by Council Regulation (EEC) No 3820/85 of 20 December 1985 on the harmonisation of certain social legislation relating to road transport (1); use of the recording equipment as defined by Council Regulation (EEC) No 3821/85 of 20 December 1985 on recording equipment in road transport (2), 4.1.2. Rules concerning the type of transport concerned: goods or passengers; 4.1.3. Vehicle and transport documents required for the national and international carriage of goods and passengers; 4.1.4. How to behave in the event of an accident; knowledge of measures to be taken after an accident or similar occurrence, including emergency action such as evacuation of passengers and basic knowledge of first aid; 4.1.5. The precautions to be taken during the removal and replacement of wheels; 4.1.6. Rules on vehicle weights and dimensions; rules on speed limiters; 4.1.7. Obstruction of the field of view caused by the characteristics of their vehicles; 4.1.8. Reading a road map, route planning, including the use of electronic navigation systems (optional); 4.1.9. Safety factors relating to vehicle loading: controlling the load (stowing and fastening), difficulties with different kinds of load (e.g. liquids, hanging loads, ¦), loading and unloading goods and the use of loading equipment (categories C, CE, C1, C1E only); 4.1.10. The driver's responsibility in respect to the carriage of passengers; comfort and safety of passengers; transport of children; necessary checks before driving away; all sorts of buses should be part of the theory test (public service buses and coaches, buses with special dimensions, ¦) (categories D, DE, D1, D1E only). 4.2. Compulsory check of general knowledge on the following additional provisions concerning categories C, CE, D and DE: 4.2.1. The principles of the construction and functioning of: internal combustion engines, fluids (e.g. engine oil, coolant, washer fluid), the fuel system, the electrical system, the ignition system, the transmission system (clutch, gearbox, etc.); 4.2.2. Lubrication and antifreeze protection; 4.2.3. The principles of the construction, the fitting, correct use and care of tyres; 4.2.4. The principles of the types, operation, main parts, connection, use and day-to-day maintenance of brake fittings and speed governors, and use of anti-lock brakes; 4.2.5. The principles of the types, operation, main parts, connection, use and day-to-day maintenance of coupling systems (categories CE, DE only); 4.2.6. Methods of locating causes of breakdowns; 4.2.7. Preventive maintenance of vehicles and necessary running repairs; 4.2.8. The driver's responsibility in respect of the receipt, carriage and delivery of goods in accordance with the agreed conditions (categories C, CE only). B. TEST OF SKILLS AND BEHAVIOUR 5. The vehicle and its equipment 5.1. The driving of a vehicle with manual transmission shall be subject to the passing of a skills and behaviour test taken on a vehicle with manual transmission. If an applicant takes the test of skills and behaviour on a vehicle with automatic transmission this shall be recorded on any licence issued on the basis of such a test. Licences with this indication shall be used only for driving vehicles with automatic transmission. Vehicle with automatic transmission means a vehicle in which the gear ratio between the engine and the wheels can be varied by use only of the accelerator or the brakes 5.2. The vehicles used in tests of skills and behaviour shall comply with the minimum criteria given below. Member States may make provisions for more stringent criteria or add others. Category A1: Category A1 motorcycle without sidecar, with a cubic capacity of at least 120 cm3, and capable of a speed of at least 90 km/h Category A2: Motorcycle without sidecar, with a cylinder capacity of at least 400 cm3, and an engine power of at least 25 kW Category A Motorcycle without sidecar, with a cylinder capacity of at least 600 cm3, and an engine power of at least 40 kW Category B: A four-wheeled category B vehicle capable of a speed of at least 100 km/h; Category BE: A combination, made up of a category B test vehicle and a trailer with a maximum authorised mass of at least 1 000 kg, capable of a speed of at least 100 km/h, which does not fall within category B; the cargo compartment of the trailer shall consist of a closed box body which is at least as wide and as high as the motor vehicle; the closed box body may also be slightly less wide than the motor vehicle provided that the view to the rear is only possible by use of the external rear-view mirrors of the motor vehicle; the trailer shall be presented with a minimum of 800 kg real total mass; Category B1: A motor-powered quadricycle capable of a speed of at least 60 km/h; Category C: A category C vehicle with a maximum authorised mass of at least 12 000 kg, a length of at least 8 m, a width of at least 2,40 m and capable of a speed of at least 80 km/h; fitted with anti-lock brakes, equipped with a gearbox having at least eight forward ratios and recording equipment as defined by Regulation (EEC) No 3821/85; the cargo compartment shall consist of a closed box body which is at least as wide and as high as the cab; the vehicle shall be presented with a minimum of 10 000 kg real total mass; Category CE: either an articulated vehicle or a combination of a category C test vehicle and a trailer of at least 7,5 m in length; both the articulated vehicle and the combination shall have a maximum authorised mass of at least 20 000 kg, a length of at least 14 m and a width of at least 2,40 m, shall be capable of a speed of at least 80 km/h, fitted with anti-lock brakes, equipped with a gearbox having at least eight forward ratios and with recording equipment as defined by Regulation (EEC) No 3821/85; the cargo compartment shall consist of a closed box body which is at least as wide and as high as the cab; both the articulated vehicle and the combination shall be presented with a minimum of 15 000 kg real total mass; Category C1: A subcategory C1 vehicle with a maximum authorised mass of at least 4 000 kg, with a length of at least 5 m and capable of a speed of at least 80 km/h; fitted with anti-lock brakes and equipped with recording equipment as defined by Regulation (EEC) No 3821/85; the cargo compartment shall consist of a closed box body which is at least as wide and as high as the cab; Category C1E: A combination made up of a subcategory C1 test vehicle and a trailer with a maximum authorised mass of at least 1 250 kg; this combination shall be at least 8 m in length and capable of a speed of at least 80 km/h; the cargo compartment of the trailer shall consist of a closed box body which is at least as wide and as high as the cab; the closed box body may also be slightly less wide than the cab provided that the view to the rear is only possible by use of the external rear-view mirrors of the motor vehicle; the trailer shall be presented with a minimum of 800 kg real total mass; Category D: A category D vehicle with a length of at least 10 m, a width of at least 2,40 m and capable of a speed of at least 80 km/h; fitted with anti-lock brakes and equipped with recording equipment as defined by Regulation (EEC) No 3821/85; Category DE: A combination made up of a category D test vehicle and a trailer with a maximum authorised mass of at least 1 250 kg, a width of at least 2,40 m and capable of a speed of at least 80 km/h; the cargo compartment of the trailer shall consist of a closed box body which is at least 2 m wide and 2 m high; the trailer shall be presented with a minimum of 800 kg real total mass; Category D1: A subcategory D1 vehicle with a maximum authorised mass of at least 4 000 kg, with a length of at least 5 m and capable of a speed of at least 80 km/h; fitted with anti-lock brakes and equipped with recording equipment as defined by Regulation (EEC) No 3821/85; Category D1E: A combination made up of a subcategory D1 test vehicle and a trailer with a maximum authorised mass of at least 1 250 kg and capable of a speed of at least 80 km/h; the cargo compartment of the trailer shall consist of a closed box body which is at least 2 m wide and 2 m high; the trailer shall be presented with a minimum of 800 kg real total mass; Testing vehicles for categories BE, C, CE, C1, C1E, D, DE, D1 and D1E which are not in conformity with the minimum criteria given above but which were in use on or before the moment of entry into force of this Directive, may still be used for a period not exceeding ten years after that date. The requirements related to the load to be carried by these vehicles, may be implemented by Member States up to ten years from the moment of entry into force of Commission Directive 2000/56/EC (3). 6. Skills and behaviour to be tested concerning categories A1, A2 and A 6.1. Preparation and technical check of the vehicle with a bearing on road safety Applicants must demonstrate that they are capable of preparing to ride safely by satisfying the following requirements: 6.1.1. Adjust the protective outfit, such as gloves, boots, clothes and safety helmet; 6.1.2. Perform a random check on the condition of the tyres, brakes, steering, emergency stop switch (if applicable), chain, oil levels, lights, reflectors, direction indicators and audible warning device. 6.2. Special manoeuvres to be tested with a bearing on road safety 6.2.1. Putting the motorcycle on and off its stand and moving it, without the aid of the engine, by walking alongside the vehicle; 6.2.2. Parking the motorcycle on its stand; 6.2.3. At least two manoeuvres to be executed at slow speed, including a slalom; this should allow competence to be assessed in handling of the clutch in combination with the brake, balance, vision direction and position on the motorcycle and the position of the feet on the foot rests; 6.2.4. At least two manoeuvres to be executed at higher speed, of which one manoeuvre in second or third gear, at least 30 km/h and one manoeuvre avoiding an obstacle at a minimum speed of 50 km/h; this should allow competence to be assessed in the position on the motorcycle, vision direction, balance, steering technique and technique of changing gears; 6.2.5. Braking: at least two braking exercises shall be executed, including an emergency brake at a minimum speed of 50 km/h; this should allow competence to be assessed in handling of the front and rear brake, vision direction and the position on the motorcycle. The special manoeuvres mentioned under points 6.2.3 to 6.2.5 have to be implemented at the latest five years after entry into force of Directive 2000/56/EC. 6.3. Behaviour in traffic Applicants must perform all the following actions in normal traffic situations, in complete safety and taking all necessary precautions: 6.3.1. Riding away: after parking, after a stop in traffic; exiting a driveway; 6.3.2. Riding on straight roads; passing oncoming vehicles, including in confined spaces; 6.3.3. Riding round bends; 6.3.4. Crossroads: approaching and crossing of intersections and junctions; 6.3.5. Changing direction: left and right turns; changing lanes; 6.3.6. Approach/exit of motorways or similar (if available): joining from the acceleration lane; leaving on the deceleration lane; 6.3.7. Overtaking/passing: overtaking other traffic (if possible); riding alongside obstacles, e.g. parked cars; being overtaken by other traffic (if appropriate); 6.3.8. Special road features (if available): roundabouts; railway level crossings; tram/bus stops; pedestrian crossings; riding up-/downhill on long slopes; 6.3.9. Taking the necessary precautions when getting off the vehicle. 7. Skills and behaviour to be tested concerning categories B, B1 and BE 7.1. Preparation and technical check of the vehicle with a bearing on road safety Applicants must demonstrate that they are capable of preparing to drive safely by satisfying the following requirements: 7.1.1. Adjusting the seat as necessary to obtain a correct seated position; 7.1.2. Adjusting rear-view mirrors, seat belts and head restraints if available; 7.1.3. Checking that the doors are closed; 7.1.4. Performing a random check on the condition of the tyres, steering, brakes, fluids (e.g. engine oil, coolant, washer fluid), lights, reflectors, direction indicators and audible warning device; 7.1.5. Checking the safety factors relating to vehicle loading: body, sheets, cargo doors, cabin locking, way of loading, securing load (category BE only); 7.1.6. Checking the coupling mechanism and the brake and electrical connections (category BE only). 7.2. Categories B and B1: special manoeuvres to be tested with a bearing on road safety A selection of the following manoeuvres shall be tested (at least two manoeuvres for the four points, including one in reverse gear): 7.2.1. Reversing in a straight line or reversing right or left round a corner while keeping within the correct traffic lane; 7.2.2. Turning the vehicle to face the opposite way, using forward and reverse gears; 7.2.3. Parking the vehicle and leaving a parking space (parallel, oblique or right-angle, forwards or in reverse, on the flat, uphill or downhill); 7.2.4. Braking accurately to a stop; however, performing an emergency stop is optional. 7.3. Category BE: special manoeuvres to be tested with a bearing on road safety 7.3.1. Coupling and uncoupling, or uncoupling and re-coupling a trailer from its motor vehicle; the manoeuvre must involve the towing vehicle being parked alongside the trailer (i.e. not in one line); 7.3.2. Reversing along a curve, the line of which shall be left to the discretion of the Member States; 7.3.3. Parking safely for loading/unloading. 7.4. Behaviour in traffic Applicants must perform all the following actions in normal traffic situations, in complete safety and taking all necessary precautions: 7.4.1. Driving away: after parking, after a stop in traffic; exiting a driveway; 7.4.2. Driving on straight roads; passing oncoming vehicles, including in confined spaces; 7.4.3. Driving round bends; 7.4.4. Crossroads: approaching and crossing of intersections and junctions; 7.4.5. Changing direction: left and right turns; changing lanes; 7.4.6. Approach/exit of motorways or similar (if available): joining from the acceleration lane; leaving on the deceleration lane; 7.4.7. Overtaking/passing: overtaking other traffic (if possible); driving alongside obstacles, e.g. parked cars; being overtaken by other traffic (if appropriate); 7.4.8. Special road features (if available): roundabouts; railway level crossings; tram/bus stops; pedestrian crossings; driving up-/downhill on long slopes; 7.4.9. Taking the necessary precautions when alighting from the vehicle. 8. Skills and behaviour to be tested concerning categories C, CE, C1, C1E, D, DE, D1 and D1E 8.1. Preparation and technical check of the vehicle with a bearing on road safety Applicants must demonstrate that they are capable of preparing to drive safely by satisfying the following requirements: 8.1.1. Adjusting the seat as necessary to obtain a correct seated position; 8.1.2. Adjusting rear-view mirrors, seat belts and head restraints if available; 8.1.3. Random checks on the condition of the tyres, steering, brakes, lights, reflectors, direction indicators and audible warning device; 8.1.4. Checking the power-assisted braking and steering systems; checking the condition of the wheels, wheelnuts, mudguards, windscreen, windows and wipers, fluids (e.g. engine oil, coolant, washer fluid); checking and using the instrument panel including the recording equipment as defined in Regulation (EEC) No 3821/85; 8.1.5. Checking the air pressure, air tanks and the suspension; 8.1.6. Checking the safety factors relating to vehicle loading: body, sheets, cargo doors, loading mechanism (if available), cabin locking (if available), way of loading, securing load (categories C, CE, C1, C1E only); 8.1.7. Checking the coupling mechanism and the brake and electrical connections (categories CE, C1E, DE, D1E only); 8.1.8. Being capable of taking special vehicle safety measures; controlling the body, service doors, emergency exits, first aid equipment, fire extinguishers and other safety equipment (categories D, DE, D1, D1E only); 8.1.9. Reading a road map, route planning, including the use of electronic navigation systems (optional). 8.2. Special manoeuvres to be tested with a bearing on road safety 8.2.1. Coupling and uncoupling, or uncoupling and re-coupling a trailer from its motor vehicle; the manoeuvre must involve the towing vehicle being parked alongside the trailer (i.e. not in one line) (categories CE, C1E, DE, D1E only); 8.2.2. Reversing along a curve, the line of which shall be left to the discretion of the Member States; 8.2.3. Parking safely for loading/unloading at a loading ramp/platform or similar installation (categories C, CE, C1, C1E only); 8.2.4. Parking to let passengers on or off the bus safely (categories D, DE, D1, D1E only). 8.3. Behaviour in traffic Applicants must perform all the following actions in normal traffic situations, in complete safety and taking all necessary precautions: 8.3.1. Driving away: after parking, after a stop in traffic; exiting a driveway; 8.3.2. Driving on straight roads; passing oncoming vehicles, including in confined spaces; 8.3.3. Driving round bends; 8.3.4. Crossroads: approaching and crossing of intersections and junctions; 8.3.5. Changing direction: left and right turns; changing lanes; 8.3.6. Approach/exit of motorways or similar (if available): joining from the acceleration lane; leaving on the deceleration lane; 8.3.7. Overtaking/passing: overtaking other traffic (if possible); driving alongside obstacles, e.g. parked cars; being overtaken by other traffic (if appropriate); 8.3.8. Special road features (if available): roundabouts; railway level crossings; tram/bus stops; pedestrian crossings; driving up-/downhill on long slopes; 8.3.9. Taking the necessary precautions when alighting from the vehicle. 9. Marking of the test of skills and behaviour 9.1. For each of the abovementioned driving situations, the assessment must reflect the degree of ease with which the applicant handles the vehicle controls and his demonstrated capacity to drive in traffic in complete safety. The examiner must feel safe throughout the test. Driving errors or dangerous conduct immediately endangering the safety of the test vehicle, its passengers or other road users shall be penalised by failing the test, whether or not the examiner or accompanying person has to intervene. Nonetheless, the examiner shall be free to decide whether or not the skills and behaviour test should be completed. Driving examiners must be trained to assess correctly the applicants' ability to drive safely. The work of driving examiners must be monitored and supervised, by a body authorised by the Member State, to ensure correct and consistent application of fault assessment in accordance with the standards laid down in this Annex. 9.2. During their assessment, driving examiners shall pay special attention to whether an applicant is showing a defensive and social driving behaviour. This should reflect the overall style of driving and the driving examiner should take this into account in the overall picture of the applicant. It includes adapted and determined (safe) driving, taking into account road and weather conditions, taking into account other traffic, taking into account the interests of other road users (particularly the more vulnerable) and anticipation. 9.3. The driving examiner will furthermore assess whether the applicant is: 9.3.1. Controlling the vehicle; taking into account: proper use of safety belts, rear-view mirrors, head restraints; seat; proper use of lights and other equipment; proper use of clutch, gearbox, accelerator, braking systems (including third braking system, if available), steering; controlling the vehicle under different circumstances, at different speeds; steadiness on the road; the weight and dimensions and characteristics of the vehicle; the weight and type of load (categories BE, C, CE, C1, C1E, DE, D1E only); the comfort of the passengers (categories D, DE, D1, D1E only) (no fast acceleration, smoothly driving and no hard braking); 9.3.2. Driving economically and in an environmentally friendly way, taking into account the revolutions per minute, changing gears, braking and accelerating (categories BE, C, CE, C1, C1E, D, DE, D1, D1E only); 9.3.3. Observation: all-round observation; proper use of mirrors; far, middle, near distance vision; 9.3.4. Priority/giving way: priority at crossroads, intersections and junctions; giving way at other occasions (e.g. changing direction, changing lanes, special manoeuvres); 9.3.5. Correct position on the road: proper position on the road, in lanes, on roundabouts, round bends, suitable for the type and the characteristics of the vehicle; pre-positioning; 9.3.6. Keeping distance: keeping adequate distance to the front and the side; keeping adequate distance from other road users; 9.3.7. Speed: not exceeding the maximum allowed speed; adapting speed to weather/traffic conditions and where appropriate up to national speed limits; driving at such a speed that stopping within distance of the visible and free road is possible; adapting speed to general speed of same kind of road users; 9.3.8. Traffic lights, road signs and other indications: acting correctly at traffic lights; obeying instructions from traffic controllers; acting correctly at road signs (prohibitions or commands); take appropriate action at road markings; 9.3.9. Signalling: give signals where necessary, correctly and properly timed; indicating directions correctly; taking appropriate action with regard to all signals made by other road users; 9.3.10. Braking and stopping: decelerating in time, braking or stopping according to circumstances; anticipation; using the various braking systems (only for categories C, CE, D, DE); using speed reduction systems other than the brakes (only for categories C, CE, D, DE). 10. Length of the test The length of the test and the distance travelled must be sufficient to assess the skills and behaviour laid down in paragraph B of this Annex. In no circumstances should the time spent driving on the road be less than 25 minutes for categories A, A1, A2, B, B1 and BE and 45 minutes for the other categories. This does not include the reception of the applicant, the preparation of the vehicle, the technical check of the vehicle with a bearing on road safety, the special manoeuvres and the announcement of the outcome of the practical test. 11. Location of the test The part of the test to assess the special manoeuvres may be conducted on a special testing ground. Wherever practicable, the part of the test to assess behaviour in traffic should be conducted on roads outside built-up areas, expressways and motorways (or similar), as well as on all kinds of urban streets (residential areas, 30 and 50 km/h areas, urban expressways) which should represent the various types of difficulty likely to be encountered by drivers. It is also desirable for the test to take place in various traffic density conditions. The time spent driving on the road should be used in an optimal way to assess the applicant in all the various traffic areas that can be encountered, with a special emphasis on changing between these areas. II. KNOWLEDGE, SKILL AND BEHAVIOUR FOR DRIVING A POWER-DRIVEN VEHICLE Drivers of all power-driven vehicles must at any moment have the knowledge, skills and behaviour described under points 1 to 9, with a view to be able to:  Recognise traffic dangers and assess their seriousness,  Have sufficient command of their vehicle not to create dangerous situations and to react appropriately should such situations occur,  Comply with road traffic regulations, and in particular those intended to prevent road accidents and to maintain the flow of traffic,  Detect any major technical faults in their vehicles, in particular those posing a safety hazard, and have them remedied in an appropriate fashion,  Take account of all the factors affecting driving behaviour (e.g. alcohol, fatigue, poor eyesight, etc.) so as to retain full use of the faculties needed to drive safely,  Help ensure the safety of all road users, and in particular of the weakest and most exposed by showing due respect for others. Member States may implement the appropriate measures to ensure that drivers who have lost the knowledge, skills and behaviour as described under points 1 to 9 can recover this knowledge and these skills and will continue to exhibit such behaviour required for driving a motor vehicle. (1) OJ L 370, 31.12.1985, p. 1. Regulation as repealed by Regulation (EC) No 561/2006 of the European Parliament and of the Council (OJ L 102, 11.4.2006, p. 1). (2) OJ L 370, 31.12.1985, p. 8. Regulation as last amended by Regulation (EC) No 561/2006. (3) Commission Directive 2000/56/EC of 14 September 2000 amending Council Directive 91/439/EEC on driving licences (OJ L 237, 21.9.2000, p. 45). ANNEX III MINIMUM STANDARDS OF PHYSICAL AND MENTAL FITNESS FOR DRIVING A POWER-DRIVEN VEHICLE DEFINITIONS 1. For the purpose of this Annex, drivers are classified in two groups: 1.1. Group 1: drivers of vehicles of categories A, A1, A2, AM, B, B1 and BE. 1.2. Group 2: drivers of vehicles of categories C, CE, C1, C1E, D, DE, D1 and D1E. 1.3. National legislation may provide for the provisions set out in this Annex for Group 2 drivers to apply to drivers of Category B vehicles using their driving licence for professional purposes (taxis, ambulances, etc.). 2. Similarly, applicants for a first driving licence or for the renewal of a driving licence are classified in the group to which they will belong once the licence has been issued or renewed. MEDICAL EXAMINATIONS 3. Group 1: Applicants shall be required to undergo a medical examination if it becomes apparent, when the necessary formalities are being completed or during the tests which they have to undergo prior to obtaining a driving licence, that they have one or more of the medical disabilities mentioned in this Annex. 4. Group 2: Applicants shall undergo medical examinations before a driving licence is first issued to them and thereafter drivers shall be checked in accordance with the national system in place in the Member State of normal residence whenever their driving licence is renewed 5. The standards set by Member States for the issue or any subsequent renewal of driving licences may be stricter than those set out in this Annex. SIGHT 6. All applicants for a driving licence shall undergo an appropriate investigation to ensure that they have adequate visual acuity for driving power-driven vehicles. Where there is reason to doubt that the applicant's vision is adequate, he shall be examined by a competent medical authority. At this examination attention shall be paid the following in particular: visual acuity, field of vision, twilight vision and progressive eye diseases. For the purpose of this Annex, intra-ocular lenses shall not be considered corrective lenses. Group 1: 6.1. Applicants for a driving licence or for the renewal of such a licence shall have a binocular visual acuity, with corrective lenses if necessary, of at least 0,5 when using both eyes together. Driving licences shall not be issued or renewed if, during the medical examination, it is shown that the horizontal field of vision is less than 120o o, apart from exceptional cases duly justified by a favourable medical opinion and a positive practical test, or that the person concerned suffers from any other eye condition that would compromise safe driving. When a progressive eye disease is detected or declared, driving licences may be issued or renewed subject to the applicant undergoing regular examination by a competent medical authority. 6.2. Applicants for a driving licence, or for the renewal of such a licence, who have total functional loss of vision in one eye or who use only one eye (e.g. in the case of diplopia) must have a visual acuity of at least 0,6, with corrective lenses if necessary. The competent medical authority must certify that this condition of monocular vision has existed sufficiently long to allow adaptation and that the field of vision in this eye is normal. Group 2: 6.3. Applicants for a driving licence or for the renewal of such a licence must have a visual acuity, with corrective lenses if necessary, of at least 0,8 in the better eye and at least 0,5 in the worse eye. If corrective lenses are used to attain the values of 0,8 and 0,5, the uncorrected acuity in each eye must reach 0,05, or else the minimum acuity (0,8 and 0,5) must be achieved either by correction by means of glasses with a power not exceeding plus or minus 8 dioptres or with the aid of contact lenses (uncorrected vision = 0,05). The correction must be well tolerated. Driving licences shall not be issued to or renewed for applications or drivers without a normal binocular field of vision or suffering from diplopia. HEARING 7. Driving licences may be issued to or renewed for applicants or drivers in Group 2 subject to the opinion of the competent medical authorities; particular account will be taken in medical examinations of the scope for compensation. PERSONS WITH A LOCOMOTOR DISABILITY 8. Driving licences shall not be issued to or renewed for applicants or drivers suffering from complaints or abnormalities of the locomotor system which make it dangerous to drive a power-driven vehicle. Group 1: 8.1. Driving licences subject to certain restrictions, if necessary, may be issued to physically disabled applicants or drivers following the issuing of an opinion by a competent medical authority. This opinion must be based on a medical assessment of the complaint or abnormality in question and, where necessary, on a practical test. It must also indicate what type of modification to the vehicle is required and whether the driver needs to be fitted with an orthopaedic device, insofar as the test of skills and behaviour demonstrates that with such a device driving would not to be dangerous. 8.2. Driving licences may be issued to or renewed for any applicant suffering from a progressive complaint on condition that the disabled person is regularly examined to check that the person is still capable of driving the vehicle completely safely. Where the disability is static, driving licences may be issued or renewed without the applicant being subject to regular medical examination. Group 2: 8.3. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. CARDIOVASCULAR DISEASES 9. Any disease capable of exposing an applicant for a first licence or a driver applying for renewal to a sudden failure of the cardiovascular system such that there is a sudden impairment of the cerebral functions constitutes a danger to road safety. Group 1: 9.1. Driving licences will not to be issued to, or renewed for, applicants or drivers with serious arrhythmia. 9.2. Driving licences may be issued to, or renewed for, applicants or drivers wearing a pacemaker subject to authorised medical opinion and regular medical check-ups. 9.3. The question of whether to issue or renew a licence for applicants or drivers suffering from abnormal arterial blood pressure shall be assessed with reference to the other results of the examination, any associated complications and the danger they might constitute for road safety. 9.4. Generally speaking, a driving licence shall not be issued to or renewed for applicants or drivers suffering from angina during rest or emotion. The issuing or renewal of a driving licence to any applicant or driver having suffered myocardial infarction shall be subject to authorised medical opinion and, if necessary, regular medical check-ups. Group 2: 9.5. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. DIABETES MELLITUS 10. Driving licences may be issued to, or renewed for, applicants or drivers suffering from diabetes mellitus, subject to authorised medical opinion and regular medical check-ups appropriate to each case. Group 2: 10.1. Only in very exceptional cases may driving licences be issued to, or renewed for, applicants or drivers in this group suffering from diabetes mellitus and requiring insulin treatment, and then only where duly justified by authorised medical opinion and subject to regular medical check-ups. NEUROLOGICAL DISEASES 11. Driving licences shall not be issued to, or renewed for, applicants or drivers suffering from a serious neurological disease, unless the application is supported by authorised medical opinion. Neurological disturbances associated with diseases or surgical intervention affecting the central or peripheral nervous system, which lead to sensory or motor deficiencies and affect balance and coordination, must accordingly be taken into account in relation to their functional effects and the risks of progression. In such cases, the issue or renewal of the licence may be subject to periodic assessment in the event of risk of deterioration. 12. Epileptic seizures or other sudden disturbances of the state of consciousness constitute a serious danger to road safety if they occur in a person driving a power-driven vehicle. Group 1: 12.1. A licence may be issued or renewed subject to an examination by a competent medical authority and to regular medical check-ups. The authority shall decide on the state of the epilepsy or other disturbances of consciousness, its clinical form and progress (no seizure in the last two years, for example), the treatment received and the results thereof. Group 2: 12.2. Driving licences shall not be issued to or renewed for applicants or drivers suffering or liable to suffer from epileptic seizures or other sudden disturbances of the state of consciousness. MENTAL DISORDERS Group 1: 13.1. Driving licences shall not be issued to, or renewed for, applicants or drivers who suffer from:  severe mental disturbance, whether congenital or due to disease, trauma or neurosurgical operations,  severe mental retardation,  severe behavioural problems due to ageing; or personality defects leading to seriously impaired judgment, behaviour or adaptability, unless their application is supported by authorised medical opinion and, if necessary, subject to regular medical check-ups. Group 2: 13.2. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. ALCOHOL 14. Alcohol consumption constitutes a major danger to road safety. In view of the scale of the problem, the medical profession must be very vigilant. Group 1: 14.1. Driving licences shall not be issued to, or renewed for, applicants or drivers who are dependent on alcohol or unable to refrain from drinking and driving. After a proven period of abstinence and subject to authorised medical opinion and regular medical check-ups, driving licences may be issued to, or renewed for, applicant or drivers who have in the past been dependent on alcohol. Group 2: 14.2. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. DRUGS AND MEDICINAL PRODUCTS 15. Abuse: Driving licences shall not be issued to or renewed for applicants or drivers who are dependent on psychotropic substances or who are not dependent on such substances but regularly abuse them, whatever category of licence is requested. Regular use: Group 1: 15.1. Driving licences shall not be issued to, or renewed for, applicants or drivers who regularly use psychotropic substances, in whatever form, which can hamper the ability to drive safely where the quantities absorbed are such as to have an adverse effect on driving. This shall apply to all other medicinal products or combinations of medicinal products which affect the ability to drive. Group 2: 15.2. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definitions of this group. RENAL DISORDERS Group 1: 16.1. Driving licences may be issued or renewed for applicants and drivers suffering from serious renal insufficiency subject to authorised medical opinion and regular medical check-ups. Group 2: 16.2. Save in exceptional cases duly justified by authorised medical opinion, and subject to regular medical check-ups, driving licences shall not be issued to or renewed for applicants or drivers suffering from serious and irreversible renal deficiency. MISCELLANEOUS PROVISIONS Group 1: 17.1. Subject to authorised medical opinion and, if necessary, regular medical check-ups, driving licences may be issued to or renewed for applications or drivers who have had an organ transplant or an artificial implant which affects the ability to drive. Group 2: 17.2. The competent medical authority shall give due consideration to the additional risks and dangers involved in the driving of vehicles covered by the definition of this group. 18. As a general rule, where applicants or drivers suffer from any disorder which is not mentioned in the preceding paragraph but is liable to be, or to result in, a functional incapacity affecting safety at the wheel, driving licences shall not be issued or renewed unless the application is supported by authorised medical opinion and, if necessary, subject to regular medical check-ups. ANNEX IV MINIMUM STANDARDS FOR PERSONS WHO CONDUCT PRACTICAL DRIVING TESTS 1. Competences required by a driving examiner 1.1. A person authorised to conduct practical assessments in a motor vehicle of the driving performance of a candidate must have knowledge, skills and understanding related to the topics listed in points 1.2 to 1.6. 1.2. The competences of an examiner must be relevant to assessing the performance of a candidate seeking the category of driving licence entitlement for which the driving test is being undertaken. 1.3. Knowledge and understanding of driving and assessment:  theory of driving behaviour,  hazard perception and accident avoidance,  the syllabus underpinning driving test standards,  the requirements of the driving test,  relevant road and traffic legislation, including relevant EU and national legislation and interpretative guidelines,  assessment theory and techniques,  defensive driving. 1.4. Assessment skills:  ability to observe accurately, monitor, and evaluate overall candidate performance, in particular:  correct and comprehensive recognition of dangerous situations,  accurate determination of cause and likely effect of such situations,  achievement of competence and recognition of errors,  uniformity and consistency in assessment,  assimilate information quickly and extract key points,  look ahead, identify potential problems, and develop strategies to deal with them,  provide timely and constructive feedback. 1.5. Personal driving skills:  A person authorised to conduct a practical test for a category of driving licence must be able to drive to a consistently high standard that type of motor vehicle. 1.6. Quality of service:  establish and communicate what the candidate can expect during the test,  communicate clearly, choosing content, style and language to suit the audience and context and deal with enquiries from candidates,  provide clear feedback about the test result,  treat candidates with respect and indiscriminately. 1.7. Knowledge about vehicle technique and physics:  knowledge about vehicle technique such as steering, tyres, brakes, lights, specially for motorcycles and heavy vehicles,  loading safety,  knowledge about vehicle physics such as speed, friction, dynamics, energy. 1.8. Driving in a fuel efficient and environmentally friendly way. 2. General conditions 2.1. A category B driving examiner: (a) must have held a category B licence for at least 3 years; (b) must be at least 23 years old; (c) must have successfully completed the initial qualification provided for in point 3 of this Annex and subsequently followed the quality assurance and the periodic training arrangements as provided for in point 4 of this Annex; (d) must have terminated a vocational education that leads at least to a completion of level 3 as defined by Council Decision 85/368/EEC of 16 July 1985 on the comparability of vocational training qualifications between the Member States of the European Community (1); (e) may not be active as a commercial driving instructor in a driving school simultaneously. 2.2. A driving examiner for the other categories: (a) must hold a driving licence in the category concerned or possess equivalent knowledge through adequate professional qualification; (b) must have successfully completed the initial qualification provided for in point 3 of this Annex and subsequently followed the quality assurance and the periodic training arrangements as provided for in point 4 of this Annex; (c) must have been a qualified category B driving examiner for at least 3 years; this period may be waived provided that the examiner in question can provide evidence of:  at least 5 years of driving in the category concerned, or,  a theoretical and practical assessment of driving ability of a standard higher than that needed to obtain a driving licence thus making that requirement unnecessary, (d) must have completed a vocational education that leads at least to a termination of the level 3 as defined by Decision 85/368/EEC; (e) may not be active as a commercial driving instructor in a driving school simultaneously. 2.3. Equivalences 2.3.1. Member States may authorise an examiner to conduct driving tests for categories AM, A1, A2 and A upon passing the initial qualification prescribed in point 3 for one of these categories. 2.3.2. Member States may authorise an examiner to conduct driving tests for categories C1, C, D1 and D upon passing the initial qualification prescribed in point 3 for one of these categories. 2.3.3. Member States may authorise an examiner to conduct driving tests for categories BE, C1E, CE, D1E and DE upon passing the initial qualification prescribed in point 3 for one of these categories. 3. Initial qualification 3.1. Initial training 3.1.1. Before a person may be authorised to conduct driving tests, that person must satisfactorily complete such training programme as a Member State may specify in order to have the competences set out in point 1. 3.1.2. Member States must determine whether the content of any particular training programme will relate to authorisation to conduct driving tests for one driving licence category, or more than one. 3.2. Examinations 3.2.1. Before a person may be authorised to conduct driving tests, that person must demonstrate a satisfactory standard of knowledge, understanding, skills and aptitude in respect of the subjects listed in point 1. 3.2.2. Member States shall operate an examination process that assesses, in a pedagogically appropriate manner, the competences of the person as defined under point 1, in particular point 1.4. The examination process must include both a theoretical element and a practical element. Computer-based assessment may be used where appropriate. The details concerning the nature and duration of any tests and assessments within the examination shall be at the discretion of the individual Member States. 3.2.3. Member States must determine whether the content of any particular examination will relate to authorisation to conduct driving tests for one driving licence category, or more than one. 4. Quality assurance and periodic training 4.1. Quality assurance 4.1.1. Member States shall have in place quality assurance arrangements to provide for the maintenance of standards of driving examiners. 4.1.2. Quality assurance arrangements should involve the supervision of examiners at work, their further training and re-accreditation, their continuing professional development, and by periodic review of the outcomes of the driving tests that they have conducted. 4.1.3. Member States must provide that each examiner is subject to yearly supervision making use of quality assurance arrangements listed in point 4.1.2. Moreover, the Member States must provide that each examiner is observed conducting tests once every 5 years, for a minimum period cumulatively of at least half a day, allowing the observation of several tests. When issues are identified corrective action should be put in place. The person undertaking the supervision must be a person authorised by the Member State for that purpose. 4.1.4 Member States may provide that where an examiner is authorised to conduct driving tests in more than one category, satisfying the supervision requirement in relation to tests for one category satisfies the requirement for more than one category. 4.1.5 The work of driving examination must be monitored and supervised by a body authorised by the Member State, to ensure correct and consistent application of assessment. 4.2. Periodic training 4.2.1. Member States shall provide that, in order to remain authorised, driving examiners, irrespective of the number of categories for which they are accredited, undertake:  a minimum regular periodic training of four days in total per period of two years in order to:  maintain and refresh the necessary knowledge and examining skills,  to develop new competences that have become essential for the exercise of their profession,  ensure that an examiner continues to conduct tests to a fair and uniform standard,  a minimum periodic training of at least five days in total per period of five years,  in order to develop and maintain the necessary practical driving skills. 4.2.2. Member States shall take the appropriate measures for ensuring that specific training is given promptly to those examiners that have found to be seriously malfunctioning by the quality assurance system in place. 4.2.3. The nature of periodic training may take the form of briefing, classroom training, conventional or electronic-based learning, and it may be undertaken on an individual or group basis. It may include such re-accreditation of standards as Member States consider appropriate. 4.2.4. Member States may provide that where an examiner is authorised to conduct driving tests in more than one category, satisfying the periodic training requirement in relation to tests for one category satisfies the requirement for more than one category, provided the condition set out in point 4.2.5 is satisfied. 4.2.5. Where an examiner has not conducted tests for a category within a 24-month period, the examiner shall undertake a suitable reassessment before being allowed to carry out driving tests relating to that category. That re-assessment may be undertaken as part of the requirement set out in point 4.2.1. 5. Acquired rights 5.1. Member States may allow persons authorised to conduct driving tests immediately before these provisions come into force to continue to conduct driving tests, notwithstanding that they were not authorised in accordance with the general conditions in point 2 or the initial qualification process set out in point 3. 5.2. Such examiners are nonetheless subject to the regular supervision and quality assurance arrangements set out in point 4. (1) OJ L 199, 31.7.1985, p. 56. ANNEX V MINIMUM REQUIREMENTS FOR DRIVER TRAINING AND TESTING FOR COMBINATIONS AS DEFINED IN THE SECOND SUBPARAGRAPH OF ARTICLE 4(4)(B) 1. Member States shall take the necessary measures to:  approve and supervise the training provided for in Article 7(1)(d) or,  organise the test of skills and behaviour provided for in Article 7(1)(d). 2.1. Duration of driver training  at least 7 hours. 3. Content of driver training The driver training shall cover the knowledge, skills and behaviour as described in points 2 and 7 of Annex II. Particular attention shall be paid to:  vehicle movement dynamics, safety criteria, tractor vehicle and trailer (coupling mechanism), correct loading and safety fittings; A practical component shall include the following exercises: acceleration, deceleration, reversing, braking, stopping distance, lane-changing, braking/evasive action, trailer swing, uncoupling from and re-coupling a trailer to its motor vehicle, parking;  Each training participant has to perform the practical component and shall demonstrate its skills and behaviour on public roads,  Vehicle combinations used for the training shall fall within the category of driving licence participants have applied for. 4. Duration and contents of the test of skills and behaviour The length of the test and the distance travelled must be sufficient to assess the skills and behaviour laid down in point 3. ANNEX VI MINIMUM REQUIREMENTS FOR DRIVER TRAINING AND TESTING FOR MOTORCYCLES WITHIN CATEGORY A (PROGRESSIVE ACCESS) 1. Member States shall take the necessary measures to:  approve and supervise the training provided for in Article 7(1)(c) or,  organise the test of skills and behaviour provided for in Article 7(1)(c). 2. Duration of driver training  at least 7 hours. 3. Content of driver training  The driver training shall contain all aspects covered in point 6 of Annex II.  Each participant has to perform the practical components of the training and shall demonstrate its skills and behaviour on public roads.  Motorcycles used for the training shall fall within the category of driving licence participants have applied for. 4. Duration and contents of the test of skills and behaviour The length of the test and the distance travelled must be sufficient to assess the skills and behaviour laid down in point 3 of this Annex. ANNEX VII Part A REPEALED DIRECTIVE AS SUCCESSIVELY AMENDED (referred to in Article 17) Council Directive 91/439/EEC (1) (OJ L 237, 24.8.1991, p. 1) Council Directive 94/72/EC (OJ L 337, 24.12.1994, p. 86) Council Directive 96/47/EC (OJ L 235, 17.9.1996, p. 1) Council Directive 97/26/EC (OJ L 150, 7.6.1997, p. 41) Commission Directive 2000/56/EC (OJ L 237, 21.9.2000, p. 45) Directive 2003/59/EC of the European Parliament and of the Council, only Article 10, paragraph 2 (OJ L 226, 10.9.2003, p. 4) Regulation (EC) No 1882/2003 of the European Parliament and of the Council, only Annex II, point 24 (OJ L 284, 31.10.2003, p. 1) Part B DEADLINES FOR TRANSPOSITION INTO NATIONAL LAW AND FOR APPLICATION (referred to in Article 17) Directive Deadline for transposition Date of application Directive 91/439/EEC 1st July 1994 1st July 1996 Directive 94/72/EC - 1st January1995 Decision 96/427/EC - 16 July 1996 Directive 96/47/EC 1st July 1996 >1st July 1996 Directive 97/26/EC 1st January 1998 1st January 1998 Directive 2000/56/EC 30 September 2003 30 September 2003, 30 September 2008 (Annex II, point 6.2.5) and 30 September 2013 (Annex II point 5.2) Directive 2003/59/EC 10 September 2006 10 September 2008 (passenger transport) and 10 September 2009 (goods transport) (1) Directive 91/439/EEC was also amended by the following act which has not been repealed: 1994 Act of accession. ANNEX VIII CORRELATION TABLE Directive 91/439/EEC This Directive Article 1(1), first sentence Article 1(1) first sentence Article 1(1), second sentence  - Article 1(2) Article 1(2) Article 2(1) - Article 2(2) Article 1(3) - Article 2(1) Article 1(1), second sentence Article 2(2) Article 3(1) Article 3(2) Article 3(3) Article 2(3) - Article 2(4) - Article 3(1), first subparagraph, introductory words Article 4(1), first sentence - Article 4(2), first indent - Article 4(2), second indent Article 3(1), first subparagraph, first indent Article 4(3), first indent Article 3(1), first subparagraph, second indent Article 4(4)(b), first subparagraph Article 3(1), first subparagraph, third indent Article 4(4)(b), second subparagraph Article 3(1), first subparagraph, fourth indent Article 4(4)(c) Article 3(1), first subparagraph, fifth indent Article 4(4)(f) Article 3(1), first subparagraph, sixth indent Article 4(4)(g) Article 3(1), first subparagraph, seventh indent Article 4(4)(j) Article 3(1), first subparagraph, eighth indent Article 4(4)(k) Article 3(2), first subparagraph, introductory words - Article 3(2), first subparagraph, first indent Article 4(3)(a) Article 3(2), first subparagraph, second indent Article 4(4)(a) Article 3(2), first subparagraph, third indent Article 4(4)(d) Article 3(2), first subparagraph, fourth indent Article 4(4)(e) Article 3(2), first subparagraph, fifth indent Article 4(4)(h) Article 3(2), first subparagraph, sixth indent, introductory words Article 4(4)(i) Article 3(2), first subparagraph, sixth indent, first sub-indent - Article 3(2), first subparagraph, sixth indent, second sub-indent - Article 3(3), introductory words - Article 3(3), first indent Article 4(1), third sentence Article 3(3), second indent, first subparagraph Article 4(3), second indent Article 3(3), second indent, second subparagraph - Article 3(3), third indent Article 4(3), first indent Article 3(3), fourth indent Article 4(4), first indent Article 3(3), fifth indent Article 4(4), second indent - Article 4(3) Article 3(4) - Article 3(5) - Article 3(6) Article 4(5), first sentence - Article 4(5), second sentence Article 4 Article 5 Article 5(1) Article 6(1) Article 5(1)(a) Article 6(1)(a) Article 5(1)(b) Article 6(1)(b) Article 5(2), introductory words Article 6(2), introductory words Article 5(2)(a) Article 6(2)(a) Article 5(2)(b) Article 6(2)(b) - Article 6(2)(c) - Article 6(2)(d) - Article 6(2)(e) - Article 6(2)(f) Article 5(3) - Article 5(4) Article 6(4) Article 6(1), introductory words Article 4(1), second sentence Article 6(1)(a), first indent Article 4(3)(a), third indent Article 6(1)(a), second indent Article 4(4)(a), second indent Article 6(1)(b), first indent Article 4(3)(b), second indent Article 4(3)(c), second indent Article 6(1)(b), second indent first alternative Article 4(4)(b), fifth subparagraph Article 6(1)(b), second indent second alternative Article 4(4)(c), second indent Article 6(1)(b), third indent first and second alternative Article 4(4)(g), second indent Article 6(1)(b), third indent third and fourth alternative Article 4(4)(e), third indent Article 6(1)(c), first indent first and second alternative Article 4(4)(k), second indent Article 6(1)(c), first indent third and fourth alternative Article 4(4)(i), second indent Article 6(2) Article 4(6), first subparagraph - Article 4(6), second subparagraph Article 6(3) Article 4(6), third and fourth subparagraphs Article 7(1), introductory words Article 7(1), introductory words Article 7(1)(a) Article 7(1)(a) - Article 7(1)(b) - Article 7(1)(c) - Article 7(1)(d) Article 7(1)(b) Article 7(1)(e) Article 7(2) - Article 7(3) - - Article 7(2) - Article 7(3) Article 7(4) Article 7(4) Article 7(5) Article 7(5)(a) - Article 7(5)(b) - Article 7(5)(c) - Article 7(5)(d) Article 7 a(1) - Article 7 a(2) Article 8 Article 7 b Article 9 - Article 10 Article 8 Article 11 Article 9 Article 12 Article 10 Article 13(1) - Article 13(2) Article 11 Article 14 Article 12(1) - Article 12(2) - Article 12(3) Article 15 - Article 16 Article 13 Article 17, first subparagraph - Article 17, second subparagraph - Article 18 Article 14 Article 19 Annex I - Annex Ia Annex I Annex II Annex II Annex III Annex III - Annex IV - Annex V - Annex VI
============================== "END OF DOC" ==============================
        

In [23]:
laws[laws['CELEX'] == '32015L0413']['CELEX'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['Status'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['Act_type'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['Treaty'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['act_raw_text'].iloc[0]


"13.3.2015 EN Official Journal of the European Union L 68/9 DIRECTIVE (EU) 2015/413 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 11 March 2015 facilitating cross-border exchange of information on road-safety-related traffic offences (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 91(1)(c) thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the European Economic and Social Committee (1), After consulting the Committee of the Regions, Acting in accordance with the ordinary legislative procedure (2), Whereas: (1) Improving road safety is a prime objective of the Union's transport policy. The Union is pursuing a policy to improve road safety with the objective of reducing fatalities, injuries and material damage. An important e

In [ ]:
def search_docs(query):
    celex_ids : list[str,str]
    full_doc_info : list[str]

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)

    for doc in docs:
        celex_ids.append(doc.metadata['celex'])

    celex_ids = list[Any](set(celex_ids))    

    
    full_doc_info = f""" Doc{i}:\n 
    
      {laws[laws['CELEX'] == celex_id]['act_raw_text'].iloc[0]} 

""" 

    return full_doc_info    



In [ ]:
search_docs("Drug dealing Sentences")

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("Drug dealing Sentences")

retrived = []

for i , doc in enumerate (docs):
    retrived.append([f"Doc{i}:"  doc.metadata['status','treaty','act_type','celex']])


[Document(metadata={'subject_matter': 'sources and branches of the law;  European Union law;  justice;  criminal law', 'status': 'In Force', 'legal_basis': '12002M031; 12002M034', 'additional_info': 'CNS 2001/0114', 'chunk_number': 1, 'eurovoc': 'penal code; Community law - national law; criminal procedure; penalty; drug traffic', 'treaty': 'TEU (1992)', 'act_type': 'Decision_FRAMW', 'cites': 'joint_action/1997/396; 31999Y0123%2801%29; joint_action/1998/733', 'celex': '32004F0757', 'authors': 'European Council', 'act_name': 'Council Framework Decision 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the field of illicit drug trafficking', 'document_length': 13663, 'total_chunks': 6}, page_content="11.11.2004 EN Official Journal of the European Union L 335/8 COUNCIL FRAMEWORK DECISION 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the 

In [ ]:
retrived = []

for doc in docs:
    retrived.append({doc.metadata['status','treaty']})

In [45]:
docs[0].metadata['celex']

'32004F0757'

In [ ]:
def get_celex_ids(docs):
    celex_ids = []
    for i in docs:
        celex_ids.append(docs[i].metadata['celex'])

    celex_ids = list(set(celex_ids))

    return celex_ids    

In [1]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI."),
    ("human", "Answer this question: {question}")
])

formatted = prompt.invoke({"question": "What is AI?"})
print(formatted)

messages=[SystemMessage(content='You are a helpful AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Answer this question: What is AI?', additional_kwargs={}, response_metadata={})]


In [9]:
laws[laws['CELEX'] == '31997R2046']['act_raw_text'].iloc[0]   

"Avis juridique important|31997R2046Council Regulation (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addiction Official Journal L 287 , 21/10/1997 P. 0001 - 0005COUNCIL REGULATION (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addictionTHE COUNCIL OF THE EUROPEAN UNION,Having regard to the Treaty establishing the European Community, and in particular Article 130w thereof,Having regard to the proposal from the Commission (1),Acting in accordance with the procedure laid down in Article 189c of the Treaty (2),Whereas the impact on the structures of a developing society of an economy based on the production of drugs, or which derives a substantial revenue from them, undermines a country's smooth integration into the world economy;Whereas the breakdown of social structures in developing countries due to drug consumption and the related industry is detrimental to sustainable social de